# Python Network Automation Crash Course Part II: Chapters 12-20

Sources:
- Python teaching pattern and chapter bridge: local Python Crash Course notebooks generated in this workspace.
- Twin-bridges Python course: `https://github.com/twin-bridges/python_course_mar26` at `8d23646`
- Twin-bridges Netmiko course: `https://github.com/twin-bridges/netmiko_course` at `8c31f4e`

Goal: bridge a reader who has finished Python Crash Course into network automation.
Each chapter preserves the PCC teaching logic, then translates the same idea into
network inventory, command collection, Netmiko, APIs, testing, and operational safety.

Safety note: runnable cells use simulated data by default. Real Netmiko/API examples
are included as source-map study blocks so readers can adapt them intentionally in a lab.

In [ ]:
import json
import os
import random
import tempfile
from pathlib import Path
from pprint import pprint

print("Network automation study setup complete.")

## Coverage Summary

- Chapter 12: Starting a Netmiko Project: 7 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 13: Collecting Show Commands Across a Fleet: 11 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 14: Configuration, Validation, and Rollback Thinking: 27 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 15: Generating Network Reports: 15 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 16: Inventories, YAML, JSON, and Real-World Data: 17 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 17: Network APIs: 49 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 18: Building a Network Automation Tool: 11 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 19: Credentials, Ownership, and Guardrails: 43 twin-bridges source map(s), 2 authored bridge example(s)
- Chapter 20: Operating and Deploying Automation: 55 twin-bridges source map(s), 2 authored bridge example(s)

## Chapter 12: Starting a Netmiko Project

    **Bridge from Python Crash Course**

    PCC Chapter 12 begins the first larger project by combining classes, loops, and an external library.

    **Network Automation Translation**

    1. This chapter teaches connect to a lab device, find a prompt, and disconnect cleanly in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    device dict -> ConnectHandler -> prompt/show command -> disconnect
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class1/collateral/simple_conn.py`
- `netmiko_course/class1/collateral/simple_conn_cm.py`
- `netmiko_course/class1/collateral/simple_conn_dict.py`
- `netmiko_course/class1/exercises/exercise1.py`
- `netmiko_course/class1/exercises/exercise2.py`
- `netmiko_course/class1/exercises/exercise3.py`
- `netmiko_course/class1/collateral/simple_conn_slog.py`

#### Bridge Example: Netmiko connection shape

In [ ]:
device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": "use-an-environment-variable",
}

print("ConnectHandler(**device)")
print(f"Target host: {device['host']}")

#### Bridge Example: Lab-safe connection checklist

In [ ]:
checks = ["credentials loaded", "device reachable", "prompt found", "disconnect called"]

for check in checks:
    print(f"[ ] {check}")

#### Twin-bridges source map 12.1

Source: `netmiko_course/class1/collateral/simple_conn.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler


# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

net_connect = ConnectHandler(
    device_type="cisco_ios",
    # device_type="invalid",
    host="cisco3.lasthop.io",
    username="pyclass",
    password=password,
)
print(net_connect.find_prompt())
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.1
# Source: netmiko_course/class1/collateral/simple_conn.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.2

Source: `netmiko_course/class1/collateral/simple_conn_cm.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

with ConnectHandler(**my_device) as net_connect:
    print(net_connect.find_prompt())

print("Hello")
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.2
# Source: netmiko_course/class1/collateral/simple_conn_cm.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.3

Source: `netmiko_course/class1/collateral/simple_conn_dict.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)
print(net_connect.find_prompt())
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.3
# Source: netmiko_course/class1/collateral/simple_conn_dict.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.4

Source: `netmiko_course/class1/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

net_connect = ConnectHandler(
    device_type="cisco_nxos",
    host="nxos1.lasthop.io",
    username="pyclass",
    password=password,
)

print(net_connect.find_prompt())
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.4
# Source: netmiko_course/class1/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.5

Source: `netmiko_course/class1/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
}
net_connect = ConnectHandler(**device)
print(net_connect.find_prompt())
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.5
# Source: netmiko_course/class1/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.6

Source: `netmiko_course/class1/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "nxos1.out",
}
with ConnectHandler(**device) as net_connect:
    print(net_connect.find_prompt())
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.6
# Source: netmiko_course/class1/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 12.7

Source: `netmiko_course/class1/collateral/simple_conn_slog.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `connect to a lab device, find a prompt, and disconnect cleanly`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device dict -> ConnectHandler -> prompt/show command -> disconnect`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import time
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

time.sleep(4)

net_connect = ConnectHandler(
    device_type="cisco_ios",
    host="cisco3.lasthop.io",
    username="pyclass",
    password=password,
    session_log="cisco3.out",
)
print(net_connect.find_prompt())
output = net_connect.send_command("show ip int brief")
print(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 12.7
# Source: netmiko_course/class1/collateral/simple_conn_slog.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 12: Starting a Netmiko Project Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 13: Collecting Show Commands Across a Fleet

    **Bridge from Python Crash Course**

    PCC Chapter 13 grows the project by adding repeated objects and state changes.

    **Network Automation Translation**

    1. This chapter teaches run operational commands against many devices and track results in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    device fleet -> command loop -> per-device output -> result dictionary
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class2/collateral/read_timeout/traceroute_long.py`
- `netmiko_course/class2/collateral/read_timeout/traceroute_timeout.py`
- `netmiko_course/class2/collateral/read_timeout/traceroute_working.py`
- `netmiko_course/class2/collateral/conn_mult_devices.py`
- `netmiko_course/class2/exercises/exercise2.py`
- `netmiko_course/class2/exercises/exercise1.py`
- `netmiko_course/class2/exercises/exercise3.py`
- `netmiko_course/class2/exercises/exercise4.py`
- `netmiko_course/class2/collateral/expect_str.py`
- `netmiko_course/class2/collateral/show_command.py`
- `netmiko_course/class2/collateral/netmiko_log.py`

#### Bridge Example: Collect simulated show output

In [ ]:
devices = ["edge-sw1", "edge-sw2", "core-rtr1"]
command = "show ip int brief"
results = {}

for device in devices:
    results[device] = f"{device}: simulated output for {command}"

for device, output in results.items():
    print(device, "->", output)

#### Bridge Example: Track command success

In [ ]:
results = {"edge-sw1": True, "edge-sw2": True, "core-rtr1": False}
failed = [device for device, ok in results.items() if not ok]

print(f"Successful devices: {len(results) - len(failed)}")
print(f"Failed devices: {failed}")

#### Twin-bridges source map 13.1

Source: `netmiko_course/class2/collateral/read_timeout/traceroute_long.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    # "session_log": "traceroute.out",
}

command = "traceroute 8.8.8.8"

with ConnectHandler(**device) as ssh_conn:
    try:
        start_time = datetime.now()
        output = ssh_conn.send_command(command, read_timeout=60 * 1)
        # output = ssh_conn.send_command(command)
        print(output)
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.1
# Source: netmiko_course/class2/collateral/read_timeout/traceroute_long.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.2

Source: `netmiko_course/class2/collateral/read_timeout/traceroute_timeout.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    # "session_log": "traceroute.out",
}

command = "traceroute 8.8.8.8"

with ConnectHandler(**device) as ssh_conn:
    try:
        start_time = datetime.now()
        output = ssh_conn.send_command(command, read_timeout=20)
    finally:
        end_time = datetime.now()
# ... 5 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.2
# Source: netmiko_course/class2/collateral/read_timeout/traceroute_timeout.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.3

Source: `netmiko_course/class2/collateral/read_timeout/traceroute_working.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    # "session_log": "traceroute.out",
}

command = "traceroute 10.220.88.28"

with ConnectHandler(**device) as ssh_conn:
    start_time = datetime.now()
    output = ssh_conn.send_command(command, read_timeout=20)
    end_time = datetime.now()
    print()
    print("-" * 50)
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.3
# Source: netmiko_course/class2/collateral/read_timeout/traceroute_working.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.4

Source: `netmiko_course/class2/collateral/conn_mult_devices.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

nxos1 = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

arista1 = {
    "device_type": "arista_eos",
# ... 9 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.4
# Source: netmiko_course/class2/collateral/conn_mult_devices.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.5

Source: `netmiko_course/class2/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


cisco3 = {
    "device_type": "cisco_xe",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "cisco3.out",
}

with ConnectHandler(**cisco3) as net_connect:
    # Intentionally do something that will break
    start_prompt = net_connect.find_prompt()

    # Will fail
    # net_connect.send_command("disable")

    # Working with expect_string
# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.5
# Source: netmiko_course/class2/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.6

Source: `netmiko_course/class2/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, logging`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

import logging

logging.basicConfig(filename="test.log", level=logging.DEBUG)
logger = logging.getLogger("netmiko")

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


arista1 = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

arista2 = {
    "device_type": "arista_eos",
    "host": "arista2.lasthop.io",
    "username": "pyclass",
# ... 24 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.6
# Source: netmiko_course/class2/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.7

Source: `netmiko_course/class2/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, datetime.datetime, getpass.getpass, netmiko.ConnectHandler, netmiko.ReadTimeout`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from datetime import datetime
from getpass import getpass
from netmiko import ConnectHandler, ReadTimeout

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
}

try:
    ssh_conn = ConnectHandler(**device)
    start = datetime.now()
    show_tech = ssh_conn.send_command("show tech-support", read_timeout=5)
except ReadTimeout:
    end = datetime.now()
    print("\nProgram failed with ReadTimeout Exception.\n")
    print(f"Execution time: {end - start}")
    print("\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.7
# Source: netmiko_course/class2/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.8

Source: `netmiko_course/class2/exercises/exercise4.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, datetime.datetime, getpass.getpass, netmiko.ConnectHandler, netmiko.ReadTimeout`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from datetime import datetime
from getpass import getpass
from netmiko import ConnectHandler, ReadTimeout

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
}

try:
    ssh_conn = ConnectHandler(**device)
    start = datetime.now()
    show_tech = ssh_conn.send_command("show tech-support", read_timeout=180)
    print("\nCommand Succeeded.\n")
except ReadTimeout:
    print("\nProgram failed with ReadTimeout Exception.\n")
finally:
    end = datetime.now()
# ... 13 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.8
# Source: netmiko_course/class2/exercises/exercise4.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.9

Source: `netmiko_course/class2/collateral/expect_str.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)

output = net_connect.send_command("show ip int brief", expect_string=r"#")
print(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.9
# Source: netmiko_course/class2/collateral/expect_str.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.10

Source: `netmiko_course/class2/collateral/show_command.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)

output = net_connect.send_command("show ip int brief")
print(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.10
# Source: netmiko_course/class2/collateral/show_command.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 13.11

Source: `netmiko_course/class2/collateral/netmiko_log.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, logging`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `run operational commands against many devices and track results`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`device fleet -> command loop -> per-device output -> result dictionary`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

import logging

logging.basicConfig(filename="test.log", level=logging.DEBUG)
logger = logging.getLogger("netmiko")

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)
print(net_connect.find_prompt())
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 13.11
# Source: netmiko_course/class2/collateral/netmiko_log.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 13: Collecting Show Commands Across a Fleet Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 14: Configuration, Validation, and Rollback Thinking

    **Bridge from Python Crash Course**

    PCC Chapter 14 adds user experience, scoring, levels, and reset behavior to a project.

    **Network Automation Translation**

    1. This chapter teaches prepare config sets, apply changes, verify state, and plan rollback in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    intended state -> config commands -> verification command -> rollback notes
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class12/collateral/commit.py`
- `netmiko_course/class4/collateral/send_multiline/file_delete_pattern.py`
- `netmiko_course/class4/collateral/send_multiline/mline_pattern.py`
- `netmiko_course/class4/collateral/send_multiline_timing/file_delete_timing.py`
- `netmiko_course/class4/collateral/send_multiline_timing/mline_time.py`
- `netmiko_course/class5/collateral/config_file.py`
- `netmiko_course/class5/collateral/config_vlans.py`
- `netmiko_course/class5/exercises/exercise3.py`
- `netmiko_course/class12/exercises/exercise3.py`
- `netmiko_course/class4/exercises/exercise3.py`
- `netmiko_course/class4/exercises/exercise4.py`
- `netmiko_course/class5/exercises/exercise1.py`
- `netmiko_course/class5/exercises/exercise2.py`
- `netmiko_course/class12/exercises/exercise1.py`
- `netmiko_course/class12/exercises/exercise2.py`
- `netmiko_course/class4/exercises/exercise1.py`
- `netmiko_course/class4/exercises/exercise2.py`
- `netmiko_course/class5/collateral/disable_cmd_verify.py`
- `netmiko_course/class5/exercises/vlans.txt`
- `netmiko_course/class12/collateral/config_mode.py`
- `netmiko_course/class12/collateral/enable.py`
- `netmiko_course/class12/collateral/fast_cli.py`
- `netmiko_course/class4/collateral/send_command_prompting.py`
- `netmiko_course/class4/collateral/send_command_timing_prompting.py`
- `netmiko_course/class5/collateral/config_rm_user.py`
- `netmiko_course/class5/collateral/lab_devices.yml`
- `netmiko_course/class5/collateral/vlans.txt`

#### Bridge Example: Build VLAN config

In [ ]:
vlan = {"id": 20, "name": "USERS"}
commands = [f"vlan {vlan['id']}", f"name {vlan['name']}"]

print("Config plan:")
for command in commands:
    print(command)

#### Bridge Example: Verification plan

In [ ]:
change = {"device": "edge-sw1", "vlan": 20, "expected_name": "USERS"}
verify_command = f"show vlan id {change['vlan']}"
rollback_commands = [f"no vlan {change['vlan']}"]

print(verify_command)
print(rollback_commands)

#### Twin-bridges source map 14.1

Source: `netmiko_course/class12/collateral/commit.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


if __name__ == "__main__":

    vmx2 = {
        "device_type": "juniper_junos",
        "host": "vmx2.lasthop.io",
        "username": "pyclass",
        "password": password,
        "session_log": "output.txt",
    }

    with ConnectHandler(**vmx2) as net_connect:

        print("\n\nSend configuration commands to device:")
        cfg_commands = [
            "set system syslog archive size 110k files 3",
            "set system time-zone America/New_York",
# ... 21 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.1
# Source: netmiko_course/class12/collateral/commit.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.2

Source: `netmiko_course/class4/collateral/send_multiline/file_delete_pattern.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from getpass import getpass
from netmiko import ConnectHandler

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": getpass(),
    "device_type": "cisco_xe",
}

with ConnectHandler(**device) as net_connect:

    # cisco3#del flash:/ex1.cfg
    # Delete filename [ex1.cfg]?
    # Delete bootflash:/ex1.cfg? [confirm]y
    # cisco3#

    filename = "test_cf.txt"
    cmd_list = [
        [f"del flash:/{filename}", r"Delete filename"],
        ["\n", r"confirm"],
        ["y", r"#"],
    ]

# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.2
# Source: netmiko_course/class4/collateral/send_multiline/file_delete_pattern.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.3

Source: `netmiko_course/class4/collateral/send_multiline/mline_pattern.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler, logging`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
cisco3#traceroute
Protocol [ip]:
Target IP address: 10.220.88.28
Ingress traceroute [n]:
Source address or interface: 10.220.88.22
DSCP Value [0]:
Numeric display [n]: y
Timeout in seconds [3]:
Probe count [3]:
Minimum Time to Live [1]:
Maximum Time to Live [30]:
Port Number [33434]:
Loose, Strict, Record, Timestamp, Verbose[none]:
Type escape sequence to abort.
Tracing the route to 10.220.88.28
VRF info: (vrf in name/id, vrf out name/id)
"""
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

import logging
# ... 45 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.3
# Source: netmiko_course/class4/collateral/send_multiline/mline_pattern.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.4

Source: `netmiko_course/class4/collateral/send_multiline_timing/file_delete_timing.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from getpass import getpass
from netmiko import ConnectHandler

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": getpass(),
    "device_type": "cisco_xe",
}

with ConnectHandler(**device) as net_connect:

    # cisco3#del flash:/ex1.cfg
    # Delete filename [ex1.cfg]?
    # Delete bootflash:/ex1.cfg? [confirm]y
    # cisco3#

    filename = "test1_vlan.txt"
    cmd_list = [
        f"del flash:/{filename}",
        "\n",
        "y",
    ]

# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.4
# Source: netmiko_course/class4/collateral/send_multiline_timing/file_delete_timing.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.5

Source: `netmiko_course/class4/collateral/send_multiline_timing/mline_time.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
cisco3#traceroute
Protocol [ip]:
Target IP address: 10.220.88.28
Ingress traceroute [n]:
Source address or interface: 10.220.88.22
DSCP Value [0]:
Numeric display [n]: y
Timeout in seconds [3]:
Probe count [3]:
Minimum Time to Live [1]:
Maximum Time to Live [30]:
Port Number [33434]:
Loose, Strict, Record, Timestamp, Verbose[none]:
Type escape sequence to abort.
Tracing the route to 10.220.88.28
VRF info: (vrf in name/id, vrf out name/id)
"""
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
# ... 38 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.5
# Source: netmiko_course/class4/collateral/send_multiline_timing/mline_time.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.6

Source: `netmiko_course/class5/collateral/config_file.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, yaml, netmiko.ConnectHandler, time`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It defines reusable function(s): `load_devices`.
7. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
import yaml
from netmiko import ConnectHandler
import time


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


if __name__ == "__main__":

    # Code so automated tests will run properly
    # Check for environment variable, if that fails, use getpass().
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    device_dict = load_devices()

# ... 13 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.6
# Source: netmiko_course/class5/collateral/config_file.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.7

Source: `netmiko_course/class5/collateral/config_vlans.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, yaml, netmiko.ConnectHandler, time`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It defines reusable function(s): `load_devices`.
7. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
import yaml
from netmiko import ConnectHandler
import time


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


if __name__ == "__main__":

    # Code so automated tests will run properly
    # Check for environment variable, if that fails, use getpass().
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    device_dict = load_devices()

# ... 15 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.7
# Source: netmiko_course/class5/collateral/config_vlans.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.8

Source: `netmiko_course/class5/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, netmiko.ReadTimeout, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Then it sends configuration commands, which is the part that can change a real device.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler, ReadTimeout
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

nxos1 = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
    # "fast_cli": False,
    "session_log": "nxos1.out",
}

with ConnectHandler(**nxos1) as net_connect:

    print("\n\nChanging the terminal width...")
    output = net_connect.send_command("terminal width 80")

    print("\nSetting the hostname to a very long value...")
    output = net_connect.send_config_set("hostname verylonghostnamefornxos")

# ... 25 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.8
# Source: netmiko_course/class5/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.9

Source: `netmiko_course/class12/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Then it sends configuration commands, which is the part that can change a real device.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


if __name__ == "__main__":

    device = {
        "device_type": "juniper_junos",
        "host": "vmx1.lasthop.io",
        "username": "pyclass",
        "password": password,
        "session_log": "output.txt",
    }

    net_connect = ConnectHandler(**device)

    print("\n\nSend configuration commands to device:")
    cfg_commands = [
        "set system syslog archive size 110k files 3",
        "set system time-zone America/New_York",
# ... 22 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.9
# Source: netmiko_course/class12/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.10

Source: `netmiko_course/class4/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Reference ping output:

cisco3#ping
Protocol [ip]:
Target IP address: 8.8.8.8
Repeat count [5]: 100
Datagram size [100]:
Timeout in seconds [2]:
Extended commands [n]:
Sweep range of sizes [n]:
Type escape sequence to abort.
Sending 100, 100-byte ICMP Echos to 8.8.8.8, timeout is 2 seconds:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Success rate is 100 percent (100/100), round-trip min/avg/max = 1/2/4 ms
"""
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.10
# Source: netmiko_course/class4/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.11

Source: `netmiko_course/class4/exercises/exercise4.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Reference ping output:

cisco3#ping
Protocol [ip]:
Target IP address: 8.8.8.8
Repeat count [5]: 100
Datagram size [100]:
Timeout in seconds [2]:
Extended commands [n]:
Sweep range of sizes [n]:
Type escape sequence to abort.
Sending 100, 100-byte ICMP Echos to 8.8.8.8, timeout is 2 seconds:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Success rate is 100 percent (100/100), round-trip min/avg/max = 1/2/4 ms
"""
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.11
# Source: netmiko_course/class4/exercises/exercise4.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.12

Source: `netmiko_course/class5/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

nxos1 = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": False,
}
nxos2 = {
    "device_type": "cisco_nxos",
    "host": "nxos2.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": False,
}

config_cmds = ["ip domain-lookup", "ip domain-name bogus.com"]

# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.12
# Source: netmiko_course/class5/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.13

Source: `netmiko_course/class5/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

nxos1 = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": False,
}
nxos2 = {
    "device_type": "cisco_nxos",
    "host": "nxos2.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": False,
}

for device in (nxos1, nxos2):
    with ConnectHandler(**device) as net_connect:
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.13
# Source: netmiko_course/class5/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.14

Source: `netmiko_course/class12/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, datetime.datetime`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler
from datetime import datetime

password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_nxos",
    "host": "nxos2.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": True,
}

net_connect = ConnectHandler(**my_device)

try:
    # Command that send_command will fail on
    print(f"\n\nFast CLI state: {net_connect.fast_cli}")
    print(f"Global Delay Factor state: {net_connect.global_delay_factor}")
    start_time = datetime.now()
    output = net_connect.send_command("conf t")
finally:
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.14
# Source: netmiko_course/class12/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.15

Source: `netmiko_course/class12/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


if __name__ == "__main__":

    debug = True
    device = {
        "device_type": "cisco_ios",
        "host": "cisco4.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

    net_connect = ConnectHandler(**device)
    print(f"\nCurrent Prompt:\n{net_connect.find_prompt()}\n")

    print("Enter into configuration mode:")
    output = net_connect.config_mode()
    net_connect.clear_buffer()
# ... 12 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.15
# Source: netmiko_course/class12/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.16

Source: `netmiko_course/class4/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "cisco3.out",
    # If using send_command_timing, always set fast_cli=False
    "fast_cli": False,
}

net_connect = ConnectHandler(**my_device)

src_file = "testx.txt"
dest_file = "test-ktb.txt"
copy_cmd = f"copy flash:/{src_file} flash:/{dest_file}"

output = net_connect.send_command_timing(
# ... 14 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.16
# Source: netmiko_course/class4/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.17

Source: `netmiko_course/class4/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "cisco3.out",
}

net_connect = ConnectHandler(**my_device)

src_file = "testx.txt"
dest_file = "test-ktb.txt"
copy_cmd = f"copy flash:/{src_file} flash:/{dest_file}"

output = net_connect.send_command(
    command_string=copy_cmd,
    expect_string=r"Destination filename",
# ... 17 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.17
# Source: netmiko_course/class4/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.18

Source: `netmiko_course/class5/collateral/disable_cmd_verify.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, yaml, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Then it sends configuration commands, which is the part that can change a real device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It defines reusable function(s): `load_devices`.
7. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
import yaml
from netmiko import ConnectHandler


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


if __name__ == "__main__":

    # Code so automated tests will run properly
    # Check for environment variable, if that fails, use getpass().
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    device_dict = load_devices()

    arista1 = device_dict["arista1"]
# ... 11 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.18
# Source: netmiko_course/class5/collateral/disable_cmd_verify.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.19

Source: `netmiko_course/class5/exercises/vlans.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
vlan 501
  name blue501
vlan 502
  name blue502
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.19
# Source: netmiko_course/class5/exercises/vlans.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.20

Source: `netmiko_course/class12/collateral/config_mode.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


if __name__ == "__main__":

    debug = True
    cisco4 = {
        "device_type": "cisco_ios",
        "host": "cisco4.lasthop.io",
        "username": "pyclass",
        "password": password,
    }
    vmx2 = {
        "device_type": "juniper_junos",
        "host": "vmx2.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

# ... 22 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.20
# Source: netmiko_course/class12/collateral/config_mode.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.21

Source: `netmiko_course/class12/collateral/enable.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `lower_privileges, show_current_priv`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


def lower_privileges(net_connect):
    net_connect.exit_enable_mode()
    print("*" * 40)
    print()
    print(f"Current Prompt:\n{net_connect.find_prompt()}\n")
    show_current_priv(net_connect)


def show_current_priv(net_connect):
    output = net_connect.send_command("show priv")
    print(f"Current Privilege Level: {output}")


if __name__ == "__main__":

    my_device = {
# ... 33 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.21
# Source: netmiko_course/class12/collateral/enable.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.22

Source: `netmiko_course/class12/collateral/fast_cli.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
    "password": password,
    "fast_cli": True,
}

with ConnectHandler(**my_device) as net_connect:

    print(f"\n\n{net_connect.find_prompt()}\n\n")
    print(f"Fast CLI state: {net_connect.fast_cli}")
    print(f"Global Delay Factor state: {net_connect.global_delay_factor}")

    output = net_connect.send_command("show ip int brief")
    print()
    print("-" * 20)
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.22
# Source: netmiko_course/class12/collateral/fast_cli.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.23

Source: `netmiko_course/class4/collateral/send_command_prompting.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "cisco3.out",
}

with ConnectHandler(**my_device) as net_connect:

    filename = "file_ex_3.txt"
    cmd = f"del flash:/{filename}"

    output = net_connect.send_command(
        cmd, expect_string=r"Delete filename", strip_prompt=False, strip_command=False
    )
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.23
# Source: netmiko_course/class4/collateral/send_command_prompting.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.24

Source: `netmiko_course/class4/collateral/send_command_timing_prompting.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "cisco3.out",
}

with ConnectHandler(**my_device) as net_connect:

    filename = "cisco3-cfg-May-16-11-15-40.259-113"
    cmd = f"del flash:/{filename}"

    output = net_connect.send_command_timing(
        cmd, strip_prompt=False, strip_command=False
    )
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.24
# Source: netmiko_course/class4/collateral/send_command_timing_prompting.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.25

Source: `netmiko_course/class5/collateral/config_rm_user.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, yaml, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It defines reusable function(s): `load_devices`.
7. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
import yaml
from netmiko import ConnectHandler


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


if __name__ == "__main__":

    # Code so automated tests will run properly
    # Check for environment variable, if that fails, use getpass().
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    device_dict = load_devices()

    cisco3 = device_dict["cisco3"]
# ... 18 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.25
# Source: netmiko_course/class5/collateral/config_rm_user.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.26

Source: `netmiko_course/class5/collateral/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
  host: arista3.lasthop.io
# ... 26 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.26
# Source: netmiko_course/class5/collateral/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 14.27

Source: `netmiko_course/class5/collateral/vlans.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `prepare config sets, apply changes, verify state, and plan rollback`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`intended state -> config commands -> verification command -> rollback notes`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
vlan 500
 name gold500
vlan 501
 name gold501
vlan 502
 name gold502
vlan 503
 name gold503
vlan 504
 name gold504
```

In [ ]:
# Practice rewrite for twin-bridges source map 14.27
# Source: netmiko_course/class5/collateral/vlans.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 14: Configuration, Validation, and Rollback Thinking Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 15: Generating Network Reports

    **Bridge from Python Crash Course**

    PCC Chapter 15 turns generated data into visual insight.

    **Network Automation Translation**

    1. This chapter teaches summarize interface status, VLAN counts, latency, and compliance in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    raw command output -> parsed records -> aggregate -> report
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class3/exercises/show_vlan.ttp`
- `netmiko_course/class3/collateral/show_genie.py`
- `netmiko_course/class3/collateral/show_ttp.py`
- `netmiko_course/class3/exercises/exercise5.py`
- `netmiko_course/class3/exercises/exercise1.py`
- `netmiko_course/class3/exercises/exercise2.py`
- `netmiko_course/class3/exercises/exercise3.py`
- `netmiko_course/class3/exercises/exercise4.py`
- `netmiko_course/class3/collateral/read_timeout_timing/traceroute_timeout.py`
- `netmiko_course/class3/collateral/read_timeout_timing/traceroute_working.py`
- `netmiko_course/class3/collateral/show_genie_nxos.py`
- `netmiko_course/class3/collateral/show_run_intf.ttp`
- `netmiko_course/class3/collateral/show_textfsm.py`
- `netmiko_course/class3/collateral/show_timing.py`
- `python_course_mar26/class3/list_comprehenson/list_comp_ex.py`

#### Bridge Example: Summarize interface states

In [ ]:
interfaces = [
    {"name": "Gi1/0/1", "status": "connected"},
    {"name": "Gi1/0/2", "status": "notconnect"},
    {"name": "Gi1/0/3", "status": "connected"},
]

connected = [intf for intf in interfaces if intf["status"] == "connected"]
print(f"Connected interfaces: {len(connected)} / {len(interfaces)}")

#### Bridge Example: Compliance score

In [ ]:
checks = {"ntp": True, "syslog": True, "aaa": False, "snmp": True}
score = sum(checks.values()) / len(checks) * 100

print(f"Baseline compliance: {score:.0f}%")

#### Twin-bridges source map 15.1

Source: `netmiko_course/class3/exercises/show_vlan.ttp`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
<vars>
# Definition of port field regular expression
PORT = "\S+.*"
</vars>

<group>
{{ vlan_id | DIGIT }} {{ vlan_name }} {{ vlan_status }} {{ ports | re("PORT") }}
</group>
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.1
# Source: netmiko_course/class3/exercises/show_vlan.ttp
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.2

Source: `netmiko_course/class3/collateral/show_genie.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, pprint.pprint`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass
from pprint import pprint

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_xe",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

with ConnectHandler(**my_device) as net_connect:
    output = net_connect.send_command("show ip int brief", use_genie=True)
    # output = net_connect.send_command("show ip arp", use_genie=True)
    pprint(output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.2
# Source: netmiko_course/class3/collateral/show_genie.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.3

Source: `netmiko_course/class3/collateral/show_ttp.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, pprint.pprint`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass
from pprint import pprint

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)

output = net_connect.send_command(
    "show run", use_ttp=True, ttp_template="show_run_intf.ttp"
)
pprint(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.3
# Source: netmiko_course/class3/collateral/show_ttp.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.4

Source: `netmiko_course/class3/exercises/exercise5.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, pprint.pprint`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass
from pprint import pprint

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)
output = net_connect.send_command(
    "show vlan", use_ttp=True, ttp_template="show_vlan.ttp"
)
net_connect.disconnect()

print()
print("VLAN Table:")
# ... 12 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.4
# Source: netmiko_course/class3/exercises/exercise5.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.5

Source: `netmiko_course/class3/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


cisco3 = {
    "device_type": "cisco_xe",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

cisco4 = {
    "device_type": "cisco_xe",
    "host": "cisco4.lasthop.io",
    "username": "pyclass",
    "password": password,
}

for device in (cisco3, cisco4):
    with ConnectHandler(**device) as net_connect:
# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.5
# Source: netmiko_course/class3/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.6

Source: `netmiko_course/class3/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    "session_log": "traceroute.out",
}

command = "show tech-support"
ssh_conn = ConnectHandler(**device)

# Gather the entire output
start_time = datetime.now()
output = ssh_conn.send_command_timing(
    command, last_read=10, read_timeout=180, strip_prompt=False
)
# ... 10 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.6
# Source: netmiko_course/class3/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.7

Source: `netmiko_course/class3/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, pprint.pprint, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from pprint import pprint
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

arista1 = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

with ConnectHandler(**arista1) as net_connect:
    show_vlan = net_connect.send_command("show vlan", use_textfsm=True)

    print()
    print("VLAN Table:")
    print("-" * 18)
    pprint(show_vlan)
    print()

# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.7
# Source: netmiko_course/class3/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.8

Source: `netmiko_course/class3/exercises/exercise4.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
my_device = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

with ConnectHandler(**my_device) as net_connect:
    output = net_connect.send_command("show lldp neighbors detail", use_genie=True)
    output = output["interfaces"]
    for intf_name, v in output.items():
        print()
        print(f"Local Intf: {intf_name}")
        print("-" * 12)
        neighbor_dict = v["port_id"][intf_name]["neighbors"]
        for neighbor_name, neighbor_data in neighbor_dict.items():
            remote_port = neighbor_data["port_description"]
            mgmt_ip = neighbor_data["management_address_v4"]
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.8
# Source: netmiko_course/class3/exercises/exercise4.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.9

Source: `netmiko_course/class3/collateral/read_timeout_timing/traceroute_timeout.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    # "session_log": "traceroute.out",
}

command = "traceroute 8.8.8.8"

with ConnectHandler(**device) as ssh_conn:
    try:
        start_time = datetime.now()
        # output = ssh_conn.send_command_timing(command)
        output = ssh_conn.send_command_timing(command, last_read=5, read_timeout=15)
    finally:
# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.9
# Source: netmiko_course/class3/collateral/read_timeout_timing/traceroute_timeout.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.10

Source: `netmiko_course/class3/collateral/read_timeout_timing/traceroute_working.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, datetime.datetime, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from datetime import datetime
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

device = {
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "device_type": "cisco_xe",
    # "session_log": "traceroute.out",
}

command = "traceroute 10.220.88.28"

with ConnectHandler(**device) as ssh_conn:
    start_time = datetime.now()
    output = ssh_conn.send_command_timing(command)
    end_time = datetime.now()

    print()
# ... 5 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.10
# Source: netmiko_course/class3/collateral/read_timeout_timing/traceroute_working.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.11

Source: `netmiko_course/class3/collateral/show_genie_nxos.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, pprint.pprint`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass
from pprint import pprint

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_nxos",
    "host": "nxos1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

with ConnectHandler(**my_device) as net_connect:
    output = net_connect.send_command("show version", use_genie=True)
    pprint(output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.11
# Source: netmiko_course/class3/collateral/show_genie_nxos.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.12

Source: `netmiko_course/class3/collateral/show_run_intf.ttp`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
interface {{ interface }}
 ip address {{ ip }} {{ mask }}
 description {{ description }}
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.12
# Source: netmiko_course/class3/collateral/show_run_intf.ttp
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.13

Source: `netmiko_course/class3/collateral/show_textfsm.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass, pprint.pprint`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass
from pprint import pprint

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)

output = net_connect.send_command("show ip int brief", use_textfsm=True)
pprint(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.13
# Source: netmiko_course/class3/collateral/show_textfsm.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.14

Source: `netmiko_course/class3/collateral/show_timing.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
# Check for environment variable, if that fails, use getpass().
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**my_device)

output = net_connect.send_command_timing("show ip int brief")
print(output)
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.14
# Source: netmiko_course/class3/collateral/show_timing.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 15.15

Source: `python_course_mar26/class3/list_comprehenson/list_comp_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `rich.print`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `summarize interface status, VLAN counts, latency, and compliance`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`raw command output -> parsed records -> aggregate -> report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
from rich import print

my_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

squares = [x**2 for x in my_list]
print(squares)

cubes = [x**3 for x in my_list]
print(cubes)

evens = [x for x in my_list if x % 2 == 0]
print(evens)

odds = [x for x in my_list if x % 2 == 1]
print(odds)

sentence = "This is a test sentence."
words = sentence.split()
print(words)

capital_words = [word.upper() for word in words]
print(capital_words)
```

In [ ]:
# Practice rewrite for twin-bridges source map 15.15
# Source: python_course_mar26/class3/list_comprehenson/list_comp_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 15: Generating Network Reports Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 16: Inventories, YAML, JSON, and Real-World Data

    **Bridge from Python Crash Course**

    PCC Chapter 16 pulls data from files and online sources.

    **Network Automation Translation**

    1. This chapter teaches load inventories and transform external data into device actions in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    YAML/JSON/source of truth -> parse -> clean -> command plan
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class10/exercises/devices.yaml`
- `netmiko_course/class10/exercises/exercise2.py`
- `netmiko_course/class10/exercises/exercise1.py`
- `netmiko_course/class10/exercises/exercise3.py`
- `netmiko_course/class10/exercises/my_hosts.txt`
- `netmiko_course/class10/collateral/detect_platform.py`
- `netmiko_course/class10/collateral/snmp_detect.py`
- `netmiko_course/class10/collateral/telnet_example.py`
- `netmiko_course/class10/collateral/write_read.py`
- `python_course_mar26/class1/exercises/dict_ex/net_object_net128.json`
- `python_course_mar26/class1/exercises/file_ex/network_objects.json`
- `python_course_mar26/class2/exercises/complex_ds_ex/show_tasks.json`
- `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex2.py`
- `python_course_mar26/class1/exercises/list_ex/list_ex1.py`
- `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.py`
- `python_course_mar26/class1/exercises/list_ex/list_ex1.md`
- `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.md`

#### Bridge Example: Use structured inventory data

In [ ]:
inventory = [
    {"hostname": "edge-sw1", "site": "dal", "role": "access"},
    {"hostname": "core-rtr1", "site": "dal", "role": "core"},
]

access_switches = [device for device in inventory if device["role"] == "access"]
print(access_switches)

#### Bridge Example: Generate commands from data

In [ ]:
desired_vlans = [
    {"id": 10, "name": "VOICE"},
    {"id": 20, "name": "USERS"},
]

commands = []
for vlan in desired_vlans:
    commands.extend([f"vlan {vlan['id']}", f"name {vlan['name']}"])

print(commands)

#### Twin-bridges source map 16.1

Source: `netmiko_course/class10/exercises/devices.yaml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
cisco3:
  device_type: cisco_xe
  hostname: cisco3.lasthop.io
cisco4:
  device_type: cisco_xe
  hostname: cisco4.lasthop.io
nxos1:
  device_type: cisco_nxos
  hostname: nxos1.lasthop.io
nxos2:
  device_type: cisco_nxos
  hostname: nxos2.lasthop.io
vmx1:
  device_type: juniper_junos
  hostname: vmx1.lasthop.io
vmx2:
  device_type: juniper_junos
  hostname: vmx2.lasthop.io
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.1
# Source: netmiko_course/class10/exercises/devices.yaml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.2

Source: `netmiko_course/class10/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, yaml, getpass.getpass, netmiko.SSHDetect, concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It defines reusable function(s): `find_device_type`.
4. It reads or writes files, which is how automation remembers inventory, commands, or reports.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import yaml
from getpass import getpass
from netmiko import SSHDetect
from concurrent.futures import ThreadPoolExecutor, as_completed


# Code so automated tests will run properly
PASSWORD = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


def find_device_type(hostname):

    base_device = {
        "device_type": "autodetect",
        "username": "pyclass",
        "password": PASSWORD,
    }

    device = base_device.copy()
    device["host"] = hostname
    guesser = SSHDetect(**device)
    best_match = guesser.autodetect()
    return (hostname, best_match)
# ... 31 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.2
# Source: netmiko_course/class10/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.3

Source: `netmiko_course/class10/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.SSHDetect, concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It defines reusable function(s): `find_device_type`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import SSHDetect
from concurrent.futures import ThreadPoolExecutor, as_completed


# Code so automated tests will run properly
PASSWORD = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()


def find_device_type(hostname):

    base_device = {
        "device_type": "autodetect",
        "username": "pyclass",
        "password": PASSWORD,
    }

    device = base_device.copy()
    device["host"] = hostname
    guesser = SSHDetect(**device)
    best_match = guesser.autodetect()
    return (hostname, best_match)

# ... 27 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.3
# Source: netmiko_course/class10/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.4

Source: `netmiko_course/class10/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It defines reusable function(s): `read_device`.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import time
from getpass import getpass
from netmiko import ConnectHandler


def read_device(net_connect, sleep=1):
    """Sleep and read channel."""
    time.sleep(sleep)
    output = net_connect.read_channel()
    print(output)
    return output


if __name__ == "__main__":

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    my_device = {
        "device_type": "cisco_ios",
        "host": "cisco3.lasthop.io",
# ... 25 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.4
# Source: netmiko_course/class10/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.5

Source: `netmiko_course/class10/exercises/my_hosts.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
cisco3.lasthop.io
cisco4.lasthop.io
nxos1.lasthop.io
nxos2.lasthop.io
vmx1.lasthop.io
vmx2.lasthop.io 
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.5
# Source: netmiko_course/class10/exercises/my_hosts.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.6

Source: `netmiko_course/class10/collateral/detect_platform.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.SSHDetect, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import SSHDetect, ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

base_device = {"device_type": "autodetect", "username": "pyclass", "password": password}

hosts = [
    "cisco3.lasthop.io",
    "nxos1.lasthop.io",
    "vmx1.lasthop.io",
    "arista1.lasthop.io",
]

for hostname in hosts:
    device = base_device.copy()
    device["host"] = hostname
    guesser = SSHDetect(**device)
    best_match = guesser.autodetect()
    # Name of the best device_type to use further
    print(best_match)

# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.6
# Source: netmiko_course/class10/collateral/detect_platform.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.7

Source: `netmiko_course/class10/collateral/snmp_detect.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.snmp_autodetect.SNMPDetect`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko.snmp_autodetect import SNMPDetect

snmp_key = os.getenv("SNMP_COMMUNITY")
if not snmp_key:
    snmp_key = getpass("Enter SNMP community: ")

# Had to change from a hostname to an IP address due to a Netmiko bug
# in the SNMP autodetect code.
my_snmp = SNMPDetect(
    "184.105.247.70",
    snmp_version="v3",
    user="pysnmp",
    auth_key=snmp_key,
    encrypt_key=snmp_key,
    auth_proto="sha",
    encrypt_proto="aes128",
)

device_type = my_snmp.autodetect()
print(f"\n{device_type}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.7
# Source: netmiko_course/class10/collateral/snmp_detect.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.8

Source: `netmiko_course/class10/collateral/telnet_example.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco1 = {
    "device_type": "cisco_ios_telnet",
    "host": "cisco1.lasthop.io",
    "username": "pyclass",
    "password": password,
}

net_connect = ConnectHandler(**cisco1)
print(net_connect.find_prompt())
net_connect.disconnect()
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.8
# Source: netmiko_course/class10/collateral/telnet_example.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.9

Source: `netmiko_course/class10/collateral/write_read.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import time
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

my_device = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "output.txt",
}

with ConnectHandler(**my_device) as net_connect:

    # Send command down the channel - don't forget the enter!
    net_connect.write_channel("show ip int brief\n")

    # You probably can't read right away - your Python program is faster than the device (so sleep)
    time.sleep(1)
    output = net_connect.read_channel()
# ... 14 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.9
# Source: netmiko_course/class10/collateral/write_read.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.10

Source: `python_course_mar26/class1/exercises/dict_ex/net_object_net128.json`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```json
{
    "uid": "6796b2f5-d449-44dc-9cf8-75b33c4ca493",
    "name": "hq_net_128",
    "type": "network",
    "domain": {
        "uid": "41e821a0-3720-11e3-aa6e-0800200c9fde",
        "name": "SMC User",
        "domain-type": "domain"
    },
    "subnet4": "172.31.128.0",
    "mask-length4": 24,
    "subnet-mask": "255.255.255.0",
    "icon": "NetworkObjects/network",
    "color": "light green"
}
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.10
# Source: python_course_mar26/class1/exercises/dict_ex/net_object_net128.json
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.11

Source: `python_course_mar26/class1/exercises/file_ex/network_objects.json`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```json
{
    "objects": [
        {
            "uid": "eff32e1a-d4ed-4df9-a6e6-398c5d88b34d",
            "name": "CP_default_Office_Mode_addresses_pool",
            "type": "network",
            "domain": {
                "uid": "41e821a0-3720-11e3-aa6e-0800200c9fde",
                "name": "SMC User",
                "domain-type": "domain"
            },
            "subnet4": "172.16.10.0",
            "mask-length4": 24,
            "subnet-mask": "255.255.255.0",
            "icon": "NetworkObjects/network",
            "color": "black"
        },
        {
            "uid": "6796b2f5-d449-44dc-9cf8-75b33c4ca493",
            "name": "hq_net_128",
            "type": "network",
            "domain": {
                "uid": "41e821a0-3720-11e3-aa6e-0800200c9fde",
                "name": "SMC User",
# ... 27 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.11
# Source: python_course_mar26/class1/exercises/file_ex/network_objects.json
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.12

Source: `python_course_mar26/class2/exercises/complex_ds_ex/show_tasks.json`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```json
{
    "tasks": [
        {
            "task-id": "d1313671-7c40-4c95-be76-a2ac451705e7",
            "task-name": "chkpnt-pod99 - is_ve-CMD",
            "status": "succeeded",
            "progress-percentage": 100,
            "suppressed": true,
            "comments": "Completed",
            "meta-info": {
                "lock": "unlocked",
                "validation-state": "ok",
                "last-modify-time": {
                    "posix": 1769728500793,
                    "iso-8601": "2026-01-30T00:15+0100"
                },
                "last-modifier": "WEB_API",
                "creation-time": {
                    "posix": 1769728500373,
                    "iso-8601": "2026-01-30T00:15+0100"
                },
                "creator": "WEB_API"
            }
        },
# ... 49 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.12
# Source: python_course_mar26/class2/exercises/complex_ds_ex/show_tasks.json
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.13

Source: `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex2.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `json, rich.print, ipdb`.
2. It defines reusable function(s): `read_json, extract_fields`.
3. It reads or writes files, which is how automation remembers inventory, commands, or reports.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import json
from rich import print
import ipdb  # noqa


def read_json(filename):
    data = None
    with open(filename) as f:
        data = json.load(f)

    return data


def extract_fields(task):
    task_id = task["task-id"]
    status = task["status"]
    lock = task["meta-info"]["lock"]

    return (task_id, status, lock)


if __name__ == "__main__":
    filename = "show_tasks.json"
    tasks_ds = read_json(filename)
# ... 9 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.13
# Source: python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.14

Source: `python_course_mar26/class1/exercises/list_ex/list_ex1.py`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `yaml, rich.print`.
3. It reads or writes files, which is how automation remembers inventory, commands, or reports.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
#!/usr/bin/env python
import yaml
from rich import print

filename = "locations.yml"

# Read in file as YAML
with open(filename) as f:
    locations = yaml.safe_load(f)

print()

# Print the list
print(locations)

# Print first element of list
print(f"First element: {locations[0]}")

# Print last element of list
print(f"Last element: {locations[-1]}")

# Print length of the list
print(f"List length: {len(locations)}")

# ... 26 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.14
# Source: python_course_mar26/class1/exercises/list_ex/list_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.15

Source: `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `json, rich.print, ipdb`.
2. It reads or writes files, which is how automation remembers inventory, commands, or reports.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import json
from rich import print
import ipdb  # noqa

with open("show_tasks.json") as f:
    tasks_ds = json.load(f)

ipdb.set_trace()
print(tasks_ds)
print(type(tasks_ds))
print(tasks_ds.keys())

# Remove outermost key
tasks = tasks_ds["tasks"]
ipdb.set_trace()
print(type(tasks))
print(len(tasks))

ipdb.set_trace()
for task in tasks:
    task_id = task["task-id"]
    status = task["status"]
    print(f"{task_id} --> {status}")
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.15
# Source: python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.16

Source: `python_course_mar26/class1/exercises/list_ex/list_ex1.md`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### List Exercise1

1. Read in the file "locations.yml" as YAML (this should return a list). Use 'yaml.safe_load(f)' to do this.
2. Print out this list.
3. Print first element of the list.
4. Print last element of the list.
5. Print length of the list.
6. Append "Leipzig" to the end of the list.
7. Change the fourth element of the list to be 'Stuttgart'.
8. Print out the current list.
9. Use list concatentation to add the following list: ["Dortmund", "Essen"]
10. Print out the current list.
11. Pop the first element of the list into a variable named 'city1'.
12. Pop the last element of the list into a variable named 'city_n'.
13. Print out 'city1', 'city_n', and current 'locations' list.
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.16
# Source: python_course_mar26/class1/exercises/list_ex/list_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 16.17

Source: `python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.md`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `load inventories and transform external data into device actions`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`YAML/JSON/source of truth -> parse -> clean -> command plan`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
### Complex Data Structures Exercise1

Read "show_tasks.json" as JSON and save this data as a variable named 'tasks_ds'.

Use ipdb and ipdb.set_trace() and print 'tasks_ds', print 'type(tasks_ds)', and print 'tasks_ds.keys()'.

Next drill into the 'tasks_ds' data structure one-level and retrieve the "tasks" key; save this as a new variable named 'tasks'. In other words: tasks = tasks_ds["tasks"].

As this point the 'tasks' variable should be a list. Verify this by using 'type(tasks)' and 'len(tasks)'.

Use a for-loop to loop over the 'tasks'. Each entry in the 'tasks' list will be a task-dictionary. From this inner dictionary retrieve the "task-id" and "status" fields.

Your loop code should look similar to the following:

'''python
for task in tasks:
    task_id = task["task-id"]
    status = task["status"]
    print(f"{task_id} --> {status}")
'''
```

In [ ]:
# Practice rewrite for twin-bridges source map 16.17
# Source: python_course_mar26/class2/exercises/complex_ds_ex/complex_ds_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 16: Inventories, YAML, JSON, and Real-World Data Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 17: Network APIs

    **Bridge from Python Crash Course**

    PCC Chapter 17 uses web APIs and response dictionaries.

    **Network Automation Translation**

    1. This chapter teaches authenticate, request network data, paginate, and process API responses in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    API endpoint -> request/session -> JSON dicts -> object/config action
    ```

    **Assigned twin-bridges examples**

    - `python_course_mar26/class4/sdk_api_query/sdk_api_query.py`
- `python_course_mar26/class4/exercises/api_pages_ex/api_pages_ex.py`
- `python_course_mar26/class4/exercises/main_project/03_mgmt_api_cfg.py`
- `python_course_mar26/class4/exercises/api_pages_ex/api_query_ex.py`
- `python_course_mar26/class4/sdk_api_query/sdk_gen_api_query.py`
- `python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.md`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.py`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.py`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.py`
- `python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.py`
- `python_course_mar26/class3/chkpnt_sdk/api_notes.txt`
- `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.md`
- `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.py`
- `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_funcs.py`
- `python_course_mar26/class4/exercises/api_pages_ex/api_pagination.md`
- `python_course_mar26/class3/chkpnt_sdk/gaia_intf.py`
- `python_course_mar26/class3/chkpnt_sdk/gaia_intf_fingerprint.py`
- `python_course_mar26/class3/chkpnt_sdk/mgmt_cfg_netobj.py`
- `python_course_mar26/class2/api/gaia_auth.py`
- `python_course_mar26/class3/exercises/class_api_ex/chkpt_api_ex.md`
- `python_course_mar26/work/clear_sessions.py`
- `python_course_mar26/class3/chkpnt_sdk/gaia_cfg_dns.py`
- `python_course_mar26/class3/chkpnt_sdk/mgmt_show_networks.py`
- `python_course_mar26/class2/api/awx_auth.py`
- `python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.py`
- `python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.md`
- `python_course_mar26/class2/gaia_ssh/api_status.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_funcs.py`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.md`
- `python_course_mar26/class3/fw_policy/fw_policy.py`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.md`
- `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.md`
- `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_funcs.py`
- `python_course_mar26/class4/exercises/main_project/tests/conftest.py`
- `python_course_mar26/lib_class4/chkpt_policy_funcs.py`
- `python_course_mar26/class3/chkpnt_sdk/fingerprints.txt`
- `python_course_mar26/lib_class4/chkpt_object_funcs.py`
- `python_course_mar26/class3/exercises/object_func_ex/object_funcs.py`
- `python_course_mar26/work/gaia_intf_work.py`
- `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_edit_rule.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.py`
- `python_course_mar26/class4/exercises/show_changes_ex/show_changes.py`
- `python_course_mar26/class3/exercises/object_func_ex/conftest.py`
- `python_course_mar26/class3/exercises/fw_policy_ex1/object_funcs.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/object_funcs.py`
- `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.md`
- `python_course_mar26/class3/exercises/object_func_ex/common_object_function.md`
- `python_course_mar26/class3/exercises/object_func_ex/object_function_tests.md`

#### Bridge Example: API response dictionary

In [ ]:
response = {
    "status_code": 200,
    "objects": [
        {"name": "web-01", "ipv4-address": "198.51.100.10"},
        {"name": "db-01", "ipv4-address": "198.51.100.20"},
    ],
}

if response["status_code"] == 200:
    for obj in response["objects"]:
        print(f"{obj['name']} -> {obj['ipv4-address']}")

#### Bridge Example: Pagination pattern

In [ ]:
total = 250
limit = 100
offsets = list(range(0, total, limit))

for offset in offsets:
    print(f"GET objects?limit={limit}&offset={offset}")

#### Twin-bridges source map 17.1

Source: `python_course_mar26/class4/sdk_api_query/sdk_api_query.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=True, context="web_api"
    )
    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        api_endpoint = "show-services-tcp"
        api_res = api_client.api_call(api_endpoint)

# ... 22 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.1
# Source: python_course_mar26/class4/sdk_api_query/sdk_api_query.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.2

Source: `python_course_mar26/class4/exercises/api_pages_ex/api_pages_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=True, context="web_api"
    )
    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        udp_services = []
        offset = 0
        while True:
# ... 24 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.2
# Source: python_course_mar26/class4/exercises/api_pages_ex/api_pages_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.3

Source: `python_course_mar26/class4/exercises/main_project/03_mgmt_api_cfg.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `cfg_std_mgmt_hosts, cfg_blocked_ips, main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print  # noqa
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
from chkpt_object_funcs import cfg_host_objects, cfg_group_object, delete_host_objects
from blocked_ip_funcs import (
    read_blocked_ips_file,
    gen_host_object,
    get_current_blocked_ips,
)
from chkpt_policy_funcs import cfg_fw_rules, install_fw_policy, extract_fw_name
from host_objects import smart_console_private, smart_console_public, ansible_server
from gen_fw_rules import gen_blockedip_fw_rules, gen_mgmt_fw_rules

DEBUG = False


def cfg_std_mgmt_hosts(api_client):
    print("[green][Mgmt API Config][/green] Configure Management Hosts")
    mgmt_host_objects = [smart_console_private, smart_console_public, ansible_server]
    cfg_host_objects(api_client, host_objects=mgmt_host_objects)


# ... 79 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.3
# Source: python_course_mar26/class4/exercises/main_project/03_mgmt_api_cfg.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.4

Source: `python_course_mar26/class4/exercises/api_pages_ex/api_query_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=True, context="web_api"
    )
    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        api_endpoint = "show-services-udp"
        api_res = api_client.api_query(api_endpoint, details_level="standard")

# ... 8 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.4
# Source: python_course_mar26/class4/exercises/api_pages_ex/api_query_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.5

Source: `python_course_mar26/class4/sdk_api_query/sdk_gen_api_query.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=True, context="web_api"
    )
    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        api_endpoint = "show-services-tcp"
        query = api_client.gen_api_query(api_endpoint, details_level="standard")

# ... 19 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.5
# Source: python_course_mar26/class4/sdk_api_query/sdk_gen_api_query.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.6

Source: `python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.md`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```markdown
### Gaia API Auth Exercise

Create a Python script that authenticates to the Gaia API in your pod.

After you have authenticated extract the session ID and reuse this session ID to execute the following API command: "show-api-versions"

Print the returned JSON from that API command to standard output.

Bonus: Do the above, but create the following functions to assist you: 'login', 'api_call', 'logout'.

The 'login' function should handle the login and return the response (or alternatively return the session ID).

The 'api_call' function should handle API calls to the Gaia API and return the response. My reference function signature looks as follows:

'''python
def api_call(base_url, endpoint, headers, payload=None, ssl_verify=False):
'''

The 'logout' function should gracefully log you out of the Gaia API.
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.6
# Source: python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.7

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, ipdb`.
2. It defines class blueprint(s): `ChkPntConfigError`.
3. It defines reusable function(s): `cfg_group_objects, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
import ipdb  # noqa


class ChkPntConfigError(Exception):
    pass


def cfg_group_objects(api_client):
    """Use mgmt API to configure network object group."""

    group_params = {
        "name": "hq_net",
        "members": [
            "hq_net_128",
            "hq_net_129",
            "hq_net_130",
            "hq_net_131",
            "hq_net_132",
            "hq_net_133",
            "hq_net_134",
# ... 50 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.7
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.8

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, ipdb`.
2. It defines class blueprint(s): `ChkPntConfigError`.
3. It defines reusable function(s): `cfg_host_objects, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
import ipdb # noqa

class ChkPntConfigError(Exception):
    pass

def cfg_host_objects(api_client):
    """Use mgmt API to configure Ansible and SmartConsole host objects."""
    smart_console_private = {
        "name": "Windows SmartConsole",
        "ipv4-address": "172.31.12.101",
        "color": "red",
    }
    smart_console_public = {
        "name": "Windows SmartConsole Public",
        "ipv4-address": "3.71.9.240",
        "color": "red",
    }
    ansible_server = {
        "name": "Ansible Server",
        "ipv4-address": "3.125.34.232",
# ... 49 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.8
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.9

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, ipdb`.
2. It defines class blueprint(s): `ChkPntConfigError`.
3. It defines reusable function(s): `cfg_net_objects, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
import ipdb  # noqa


class ChkPntConfigError(Exception):
    pass


def cfg_net_objects(api_client):
    """Use mgmt API to configure network objects."""
    hq_net_128 = {
        "name": "hq_net_128",
        "subnet": "172.31.128.0",
    }
    hq_net_129 = {
        "name": "hq_net_129",
        "subnet": "172.31.129.0",
    }
    hq_net_130 = {
        "name": "hq_net_130",
        "subnet": "172.31.130.0",
# ... 80 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.9
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.10

Source: `python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `requests, os, json, rich.print, dotenv.load_dotenv, ipdb`.
2. It defines reusable function(s): `login, api_call, logout`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import requests
import os
import json
from rich import print
from dotenv import load_dotenv
import ipdb  # noqa


def login(base_url, user, password, ssl_verify=False):

    url = base_url + "login"
    headers = {"Content-Type": "application/json"}
    login_payload = {"user": user, "password": password}

    response = requests.post(
        url, data=json.dumps(login_payload), headers=headers, verify=ssl_verify
    )
    return response


def api_call(base_url, endpoint, headers, payload=None, ssl_verify=False):

    if payload is None:
        payload = {}
# ... 42 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.10
# Source: python_course_mar26/class2/exercises/gaia_api_ex/gaia_auth_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.11

Source: `python_course_mar26/class3/chkpnt_sdk/api_notes.txt`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```text

    def __init__(self, api_client_args=None):
        self.server = api_client_args.server
        # web-api versus gaia-api
        self.context = api_client_args.context 
        self.api_version = api_client_args.api_version
        self.unsafe = api_client_args.unsafe
        self.fingerprint = api_client_args.fingerprint
        self.sid = api_client_args.sid

    # Context manager
    def __enter__(self):
    def __exit__(self, exc_type, exc_value, traceback):

    # Toggle to unsafe=False and will prompt you for fingerprint the first time.
    # Verification is hard
    def get_server_fingerprint(self):
    def check_fingerprint(self):
    def save_fingerprint_to_file(server, fingerprint, filename="fingerprints.txt"):
    def read_fingerprint_from_file(server, filename="fingerprints.txt"):

    def login(self, username, password, continue_last_session=False, domain=None, read_only=False,
    def login_with_api_key(self, api_key, continue_last_session=False, domain=None, read_only=False, payload=None):
    def login_as_root(self, domain=None, payload=None):
# ... 11 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.11
# Source: python_course_mar26/class3/chkpnt_sdk/api_notes.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.12

Source: `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.md`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
### Mgmt API Exercise

Reusing the code you created for the Gaia API authentication exercise (with slight modifications of the base_url), connect and authentication to the Mgmt API of your pod.

After authenticating connect to the "show-gateway-capabilities" endpoint and retrieve the JSON payload from this reponse.

From the returned data structure extract and print out the Supported OS Versions and also extract and print out the LightSpeed supported hardware.

Your output should look similar to the following:

'''bash
$ python mgmt_api_ex1.py 

R81 Supported OS Versions: 
--------------------
['R81', 'R81.10', 'R81.20']


LightSpeed Supported Hardware: 
--------------------
[
    'QLS250 Quantum LightSpeed',
    'QLS450 Quantum LightSpeed',
    'QLS650 Quantum LightSpeed',
# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.12
# Source: python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.13

Source: `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, mgmt_funcs.login, mgmt_funcs.api_call, mgmt_funcs.logout`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from mgmt_funcs import login, api_call, logout


if __name__ == "__main__":
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"
    endpoint = "login"
    base_url = f"https://{host}/web_api/v{api_version}/"
    headers = {"Content-Type": "application/json"}

    # This looks for a .env file and loads it
    load_dotenv()
    user = "admin"
    admin_pass = os.environ["CHKP_ADMIN"]

    # Login
    url = base_url + "login"
    session_id = login(url=url, username=user, password=admin_pass)
    headers["X-chkp-sid"] = session_id

    endpoint = "show-gateway-capabilities"
# ... 29 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.13
# Source: python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_api_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.14

Source: `python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_funcs.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `requests, json`.
2. It defines class blueprint(s): `MgmtAuthError, MgmtLogoutError`.
3. It defines reusable function(s): `login, api_call, logout`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import requests
import json


class MgmtAuthError(Exception):
    """Exception raised when the session ID is missing or expired."""

    pass


class MgmtLogoutError(Exception):
    """Raised when the API returns a failure during the logout process."""

    pass


def login(url, username, password):
    """Login and return the session_id."""
    headers = {"Content-Type": "application/json"}
    ssl_verify = False
    login_payload = {"user": username, "password": password}

    response = requests.post(
        url,
# ... 36 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.14
# Source: python_course_mar26/class3/exercises/mgmt_api_ex/mgmt_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.15

Source: `python_course_mar26/class4/exercises/api_pages_ex/api_pagination.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Pagination exercise

Using a standard SDK api_call, query "show-services-udp" use the "total", "to", and "from" fields and
the payload "offset" field to paginate through the results using api_call.

Verify that you correctly received all 96 of the UDP services.

Create a second Python script that uses "api_query" to automatically retrieve all of the objects and to handle any required pagination.

Verify "api_query" successfully retrieved all 96 of the UDP services.
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.15
# Source: python_course_mar26/class4/exercises/api_pages_ex/api_pagination.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.16

Source: `python_course_mar26/class3/chkpnt_sdk/gaia_intf.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#
# show physical interface.py
# version 1.0
#
# The purpose of this script is to show a server's physical interfaces
#
# written by: Check Point software technologies inc.
# April 2019
# modified by: Kirk Byers (Feb 2026)

import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    api_server = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.16
# Source: python_course_mar26/class3/chkpnt_sdk/gaia_intf.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.17

Source: `python_course_mar26/class3/chkpnt_sdk/gaia_intf_fingerprint.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#
# show physical interface.py
# version 1.0
#
# The purpose of this script is to show a server's physical interfaces
#
# written by: Check Point software technologies inc.
# April 2019
# modified by: Kirk Byers (Feb 2026)

import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    api_server = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.17
# Source: python_course_mar26/class3/chkpnt_sdk/gaia_intf_fingerprint.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.18

Source: `python_course_mar26/class3/chkpnt_sdk/mgmt_cfg_netobj.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    api_version = "1.8"
    no_ssl_verify = True

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=no_ssl_verify, context="web_api"
    )

    with APIClient(client_args) as api_client:
        api_client.login(username, password)

# ... 17 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.18
# Source: python_course_mar26/class3/chkpnt_sdk/mgmt_cfg_netobj.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.19

Source: `python_course_mar26/class2/api/gaia_auth.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `requests, os, json, rich.print, dotenv.load_dotenv, ipdb`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import requests
import os
import json
from rich import print
from dotenv import load_dotenv
import ipdb # noqa

if __name__ == "__main__":
    host = "chkpnt-pod99.lasthop.io"
    api_version = "1.8"
    base_url = f"https://{host}/gaia_api/v{api_version}/"
    endpoint = "login"

    # This looks for a .env file and loads it
    load_dotenv()
    user = "admin"
    admin_pass = os.environ["CHKP_ADMIN"]

    url = f"{base_url}{endpoint}"
    headers = {"Content-Type": "application/json"}
    login_payload = {"user": user, "password": admin_pass}
    ssl_verify = False

    # CheckPoint uses POST even for information retrieval operations
# ... 35 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.19
# Source: python_course_mar26/class2/api/gaia_auth.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.20

Source: `python_course_mar26/class3/exercises/class_api_ex/chkpt_api_ex.md`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```markdown
### Check Point Class Exercise

Create a class named 'ChkptAPI'.

The class should only have one method (dunder-init).

The dunder-init method should have the following signature:

'''python
    def __init__(
        self,
        host,
        username,
        password,
        mode="web_api",
        api_version=None,
        ssl_verify=False,
    ):
'''

The 'mode' variable can either be 'web_api' or 'gaia_api'.

Inside your dunder-init method, initialize the following attribites: self.host, self.username, self.password, and self.ssl_verify.

# ... 56 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.20
# Source: python_course_mar26/class3/exercises/class_api_ex/chkpt_api_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.21

Source: `python_course_mar26/work/clear_sessions.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, ipdb`.
2. It defines class blueprint(s): `chkp_exception`.
3. It defines reusable function(s): `cfg_host_object, cfg_fw_rule, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


class chkp_exception(Exception):
    pass


def cfg_host_object(api_client, host_object):

    payload = {"name": host_object["name"]}
    api_res = api_client.api_call(command="show-host", payload=payload)

    payload = host_object
    if api_res.success:
        api_res = api_client.api_call(command="set-host", payload=payload)

    else:
        api_res = api_client.api_call(command="add-host", payload=payload)

    if not api_res.success:
        msg = api_res.error_message
# ... 66 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.21
# Source: python_course_mar26/work/clear_sessions.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.22

Source: `python_course_mar26/class3/chkpnt_sdk/gaia_cfg_dns.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    api_version = "1.8"
    no_ssl_verify = True

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=no_ssl_verify, context="gaia_api"
    )

    with APIClient(client_args) as api_client:
        api_client.login(username, password)

# ... 18 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.22
# Source: python_course_mar26/class3/chkpnt_sdk/gaia_cfg_dns.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.23

Source: `python_course_mar26/class3/chkpnt_sdk/mgmt_show_networks.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=True, context="web_api"
    )
    with APIClient(client_args) as api_client:
        res = api_client.login(username, password)
        print(res)

        # api_endpoint = "show-hosts"
        api_endpoint = "show-networks"
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.23
# Source: python_course_mar26/class3/chkpnt_sdk/mgmt_show_networks.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.24

Source: `python_course_mar26/class2/api/awx_auth.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `requests, os, ipdb, dotenv.load_dotenv, rich.print`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import requests
import os
import ipdb  # noqa
from dotenv import load_dotenv
from rich import print

awx_host = "54.241.198.61"
port = "32309"
base_url = f"http://{awx_host}:{port}/api/v2/"
url = f"{base_url}tokens/"

# This looks for a .env file and loads it
load_dotenv()
user = "admin"
admin_pass = os.environ["AWX_ADMIN"]
creds = (user, admin_pass)

res = requests.post(url, auth=creds, json={"description": "Testing auth"}, verify=False)
json_resp = res.json()
token = json_resp["token"]
print()
print(url)
print(res)

# ... 18 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.24
# Source: python_course_mar26/class2/api/awx_auth.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.25

Source: `python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.py`

**What This Example Is For**

It treats network facts as structured data, so code can look up exact fields instead of reading text by eye.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, gaia_auth_ex.login, gaia_auth_ex.api_call, gaia_auth_ex.logout`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from gaia_auth_ex import login, api_call, logout


if __name__ == "__main__":
    host = "chkpnt-pod99.lasthop.io"
    api_version = "1.8"
    base_url = f"https://{host}/gaia_api/v{api_version}/"

    # This looks for a .env file and loads it
    load_dotenv()
    user = "admin"
    admin_pass = os.environ["CHKP_ADMIN"]

    response = login(base_url, user, admin_pass)
    resp_struct = response.json()
    session_id = resp_struct["sid"]

    headers = {"Content-Type": "application/json"}
    headers["X-chkp-sid"] = session_id

    # Gather and display dynamic ARP data
# ... 13 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.25
# Source: python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.26

Source: `python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.md`

**What This Example Is For**

It runs the same network task across multiple devices without waiting for one device at a time.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Gaia 'show arp' Exercise

Repeat the Gaia authentication code from the Gaia authentication exercise.

In this script, use your authenticated Gaia session to execute 'show-arp'. Retreive the ARP response from the firewall and process the ARP table.

From the ARP response, you should extract both the 'mac-address' and the 'ipv4-address'. You should then print this data to standard output.

Your output should look similar to the following:

'''python
$ python gaia_proc.py 

172.31.32.1 -> 0a:61:33:92:44:55
172.31.128.1 -> 0a:15:04:3a:87:eb
172.31.144.1 -> 0a:be:1f:c0:c1:03
172.31.145.1 -> 0a:ce:8f:94:04:c9

'''
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.26
# Source: python_course_mar26/class2/exercises/gaia_api_ex/gaia_proc.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.27

Source: `python_course_mar26/class2/gaia_ssh/api_status.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    # "session_log": "output.log",
    "secret": secret,
}

with ConnectHandler(**chkpt_fw) as ssh_conn:
    print(ssh_conn.find_prompt())

# ... 10 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.27
# Source: python_course_mar26/class2/gaia_ssh/api_status.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.28

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, chkpt_exceptions.ChkPntPolicyInstallError, rich.print, ipdb`.
2. It defines reusable function(s): `extract_fw_name, display_fw_policy, install_fw_policy, cfg_fw_rule, cfg_fw_rules`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError, ChkPntPolicyInstallError
from rich import print
import ipdb  # noqa


def extract_fw_name(fqdn):
    # Extract the fw_name from the DNS name
    if "." in fqdn:
        fw_name = fqdn.split(".")[0]
        return fw_name
    else:
        raise ValueError("Invalid firewall name: {fqdn}")


def display_fw_policy(api_client, layer="Network"):
    payload = {"name": layer}
    api_res = api_client.api_call(command="show-access-rulebase", payload=payload)
    fw_rules = api_res.data["rulebase"]
    print(fw_rules)


def install_fw_policy(api_client, policy_package="Standard", targets=None):
    """
    Install the firewall policy on firewall.
# ... 45 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.28
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.29

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Chkpnt SDK Exercise1

Connect to the Mgmt API using the Chkpnt SDK. You should read your API credentials in using the .env file and load_dotenv().

Create a function named 'cfg_host_objects'  that takes one argument (api_client). In this function configure the following three host objects:

'''python
    smart_console_private = {
        "name": "Windows SmartConsole",
        "ipv4-address": "172.31.12.101",
        "color": "red",
    }
    smart_console_public = {
        "name": "Windows SmartConsole Public",
        "ipv4-address": "3.71.9.240",
        "color": "red",
    }
    ansible_server = {
        "name": "Ansible Server",
        "ipv4-address": "3.125.34.232",
        "color": "black",
    }
'''

# ... 14 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.29
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_hostobj_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.30

Source: `python_course_mar26/class3/fw_policy/fw_policy.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, ipdb`.
2. It defines class blueprint(s): `ChkPntConfigError, ChkPntPolicyInstallError`.
3. It defines reusable function(s): `extract_fw_name, display_fw_policy, install_fw_policy, cfg_fw_policy, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
import ipdb  # noqa


class ChkPntConfigError(Exception):
    pass


class ChkPntPolicyInstallError(Exception):
    pass


def extract_fw_name(api_client):
    # Extract the fw_name from the DNS name
    fqdn = api_client.server
    if "." in fqdn:
        fw_name = fqdn.split(".")[0]
        return fw_name
    else:
        raise ValueError("Invalid firewall name: {fqdn}")

# ... 103 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.30
# Source: python_course_mar26/class3/fw_policy/fw_policy.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.31

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Chkpnt SDK Exercise3

Connect to the Mgmt API using the Chkpnt SDK. Read your credentials in using the .env file and load_dotenv().

Create a new function named 'cfg_group' that is based upon your previously created 'cfg_net_objects' function.

This function should create the following group object:

'''python
    group_params = {
        "name": "hq_net",
        "members": [
            "hq_net_128",
            "hq_net_129",
            "hq_net_130",
            "hq_net_131",
            "hq_net_132",
            "hq_net_133",
            "hq_net_134",
            "hq_net_135",
        ],
        "color": "blue",
    }

# ... 15 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.31
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/group_net_objects_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.32

Source: `python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
### Chkpnt SDK Exercise2

Connect to the Mgmt API using the Chkpnt SDK. Read your credentials in using the .env file and load_dotenv().

Create a new function named 'cfg_net_objects' that is based upon your previously created 'cfg_host_objects' function.

This function should create the following network objects:

'''python
    hq_net_128 = {
        "name": "hq_net_128",
        "subnet": "172.31.128.0",
    }
    hq_net_129 = {
        "name": "hq_net_129",
        "subnet": "172.31.129.0",
    }
    hq_net_130 = {
        "name": "hq_net_130",
        "subnet": "172.31.130.0",
    }
    hq_net_131 = {
        "name": "hq_net_131",
        "subnet": "172.31.131.0",
# ... 39 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.32
# Source: python_course_mar26/class3/exercises/chkpnt_sdk_ex/mgmt_cfg_netobj_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.33

Source: `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, chkpt_exceptions.ChkPntPolicyInstallError, rich.print, ipdb`.
2. It defines reusable function(s): `extract_fw_name, display_fw_policy, install_fw_policy, cfg_fw_rule, cfg_fw_rules`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError, ChkPntPolicyInstallError
from rich import print
import ipdb  # noqa


def extract_fw_name(fqdn):
    # Extract the fw_name from the DNS name
    if "." in fqdn:
        fw_name = fqdn.split(".")[0]
        return fw_name
    else:
        raise ValueError("Invalid firewall name: {fqdn}")


def display_fw_policy(api_client, layer="Network"):
    payload = {"name": layer}
    api_res = api_client.api_call(command="show-access-rulebase", payload=payload)
    fw_rules = api_res.data["rulebase"]
    print(fw_rules)


def install_fw_policy(api_client, policy_package="Standard", targets=None):
    """
    Install the firewall policy on firewall.
# ... 40 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.33
# Source: python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.34

Source: `python_course_mar26/class4/exercises/main_project/tests/conftest.py`

**What This Example Is For**

It checks automation logic with tests, so mistakes are caught before a script touches real infrastructure.

**Explain It Like You Are New**

1. First, it brings in helper tools: `pytest, os, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `gaia_api, mgmt_api`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import pytest
import os
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


@pytest.fixture(scope="session")
def gaia_api():

    api_server = "chkpnt-pod99.lasthop.io"
    api_version = "1.8"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=api_server, api_version=api_version, unsafe=True, context="gaia_api"
    )

    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        # Object that is passed to the tests
# ... 22 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.34
# Source: python_course_mar26/class4/exercises/main_project/tests/conftest.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.35

Source: `python_course_mar26/lib_class4/chkpt_policy_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, chkpt_exceptions.ChkPntPolicyInstallError, rich.print, ipdb`.
2. It defines reusable function(s): `extract_fw_name, display_fw_policy, install_fw_policy, cfg_fw_rule, cfg_fw_rules`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError, ChkPntPolicyInstallError
from rich import print
import ipdb  # noqa

DEBUG = False

def extract_fw_name(fqdn):
    # Extract the fw_name from the DNS name
    if "." in fqdn:
        fw_name = fqdn.split(".")[0]
        return fw_name
    else:
        raise ValueError("Invalid firewall name: {fqdn}")


def display_fw_policy(api_client, layer="Network"):
    payload = {"name": layer}
    api_res = api_client.api_call(command="show-access-rulebase", payload=payload)
    fw_rules = api_res.data["rulebase"]
    print(fw_rules)


def install_fw_policy(api_client, policy_package="Standard", targets=None):
    """
# ... 41 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.35
# Source: python_course_mar26/lib_class4/chkpt_policy_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.36

Source: `python_course_mar26/class3/chkpnt_sdk/fingerprints.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
{
    "chkpnt-pod99.lasthop.io": "C15B266CC5738613FA6BDE99AC61F535AFB45909"
}
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.36
# Source: python_course_mar26/class3/chkpnt_sdk/fingerprints.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.37

Source: `python_course_mar26/lib_class4/chkpt_object_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, ipdb`.
2. It defines reusable function(s): `cfg_object, cfg_host_object, delete_host_object, delete_host_objects, cfg_group_object, cfg_host_objects`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError
import ipdb  # noqa

DEBUG = False

def cfg_object(api_client, obj_type, obj_params, delete_obj=False):

    # Check if object already exists
    object_exists = False
    payload = {"name": obj_params["name"]}
    api_res = api_client.api_call(command=f"show-{obj_type}", payload=payload)

    if api_res.success:
        object_exists = True

    if delete_obj:
        if object_exists:
            payload = {"name": obj_params["name"]}
            api_res = api_client.api_call(command=f"delete-{obj_type}", payload=payload)
        else:
            # Nothing to do, object to delete doesn't exist
            pass
        return

# ... 44 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.37
# Source: python_course_mar26/lib_class4/chkpt_object_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.38

Source: `python_course_mar26/class3/exercises/object_func_ex/object_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `ipdb`.
2. It defines class blueprint(s): `ChkPntConfigError`.
3. It defines reusable function(s): `cfg_object, cfg_host_object, cfg_network_object, cfg_group_object`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import ipdb  # noqa


class ChkPntConfigError(Exception):
    pass


def cfg_object(api_client, obj_type, obj_params):

    # Check if object already exists
    object_exists = False
    status = ""
    payload = {"name": obj_params["name"]}
    api_res = api_client.api_call(command=f"show-{obj_type}", payload=payload)

    if api_res.success:
        object_exists = True

    if object_exists:
        # Object already exists, update parameters
        print(f"Updating {obj_type} object: {obj_params}")
        api_res = api_client.api_call(command=f"set-{obj_type}", payload=obj_params)
        status = "updated"
    else:
# ... 27 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.38
# Source: python_course_mar26/class3/exercises/object_func_ex/object_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.39

Source: `python_course_mar26/work/gaia_intf_work.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, sys, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#
# show physical interface.py
# version 1.0
#
# The purpose of this script is to show a server's physical interfaces
#
# written by: Check Point software technologies inc.
# April 2019
# modified by: Kirk Byers (Feb 2026)

import os
import sys
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    api_server = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]
# ... 35 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.39
# Source: python_course_mar26/work/gaia_intf_work.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.40

Source: `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, fw_policy_funcs.cfg_fw_rule, fw_policy_funcs.install_fw_policy`.
2. It defines reusable function(s): `main`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
from fw_policy_funcs import (
    cfg_fw_rule,
    install_fw_policy,
    display_fw_policy,
)
from object_funcs import cfg_host_object
from rich import print  # noqa
import ipdb  # noqa


def main():
    host = "chkpnt-pod99.lasthop.io"

    corp_web_server = {
        "name": "Corp Web Server",
        "ipv4-address": "172.31.144.220",
        "color": "dark green",
    }

    corp_fw_rule = {
        "layer": "Network",
# ... 31 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.40
# Source: python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.41

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_edit_rule.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, fw_policy_funcs.cfg_fw_rule, fw_policy_funcs.install_fw_policy`.
2. It defines reusable function(s): `main`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
# Edit an existing firewall rule
import os
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
from fw_policy_funcs import (
    cfg_fw_rule,
    install_fw_policy,
    display_fw_policy,
)
from object_funcs import cfg_host_object
from rich import print  # noqa
import ipdb  # noqa


def main():
    host = "chkpnt-pod99.lasthop.io"

    corp_web_server = {
        "name": "Corp Web Server",
        "ipv4-address": "172.31.144.220",
        "color": "dark green",
    }

    corp_fw_rule = {
# ... 33 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.41
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_edit_rule.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.42

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, fw_policy_funcs.cfg_fw_rule, fw_policy_funcs.install_fw_policy`.
2. It defines reusable function(s): `main`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
# Edit an existing firewall rule
import os
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
from fw_policy_funcs import (
    cfg_fw_rule,
    install_fw_policy,
    display_fw_policy,
)
from rich import print  # noqa
import ipdb  # noqa


def main():
    host = "chkpnt-pod99.lasthop.io"

    corp_web_rule = {
        "layer": "Network",
        "name": "Corp Web Server Access",
        "source": "Any",
        "destination": "Corp Web Server",
        "service": ["http", "https", "ssh"],
        "action": "Accept",
        "position": 1,
# ... 24 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.42
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.43

Source: `python_course_mar26/class4/exercises/show_changes_ex/show_changes.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs, datetime.datetime`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs
from datetime import datetime, timedelta, timezone
import ipdb  # noqa


def main():
    host = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    api_version = "1.8"
    no_ssl_verify = True

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=no_ssl_verify, context="web_api"
    )

    with APIClient(client_args) as api_client:
# ... 33 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.43
# Source: python_course_mar26/class4/exercises/show_changes_ex/show_changes.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.44

Source: `python_course_mar26/class3/exercises/object_func_ex/conftest.py`

**What This Example Is For**

It checks automation logic with tests, so mistakes are caught before a script touches real infrastructure.

**Explain It Like You Are New**

1. First, it brings in helper tools: `pytest, os, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `web_api_session`.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import pytest
import os
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


@pytest.fixture(scope="session")
def web_api_session():

    api_server = "chkpnt-pod99.lasthop.io"
    api_version = "2"

    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    client_args = APIClientArgs(
        server=api_server, api_version=api_version, unsafe=True, context="web_api"
    )

    with APIClient(client_args) as api_client:
        api_client.login(username, password)

        # Object that is passed to the tests
# ... 1 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.44
# Source: python_course_mar26/class3/exercises/object_func_ex/conftest.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.45

Source: `python_course_mar26/class3/exercises/fw_policy_ex1/object_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, ipdb`.
2. It defines reusable function(s): `cfg_host_object`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError
import ipdb  # noqa


def cfg_host_object(api_client, host_object):

    obj_type = "host"

    # Check if host already exists
    object_exists = False
    payload = {"name": host_object["name"]}
    api_res = api_client.api_call(command=f"show-{obj_type}", payload=payload)

    if api_res.success:
        object_exists = True

    if object_exists:
        # Object already exists, update parameters
        print(f"Updating {obj_type} object: {host_object}")
        api_res = api_client.api_call(command=f"set-{obj_type}", payload=host_object)
    else:
        print(f"Configuring {obj_type} object: {host_object}")
        api_res = api_client.api_call(command=f"add-{obj_type}", payload=host_object)

# ... 3 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.45
# Source: python_course_mar26/class3/exercises/fw_policy_ex1/object_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.46

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/object_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `chkpt_exceptions.ChkPntConfigError, ipdb`.
2. It defines reusable function(s): `cfg_host_object`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
from chkpt_exceptions import ChkPntConfigError
import ipdb  # noqa


def cfg_host_object(api_client, host_object):

    obj_type = "host"

    # Check if host already exists
    object_exists = False
    payload = {"name": host_object["name"]}
    api_res = api_client.api_call(command=f"show-{obj_type}", payload=payload)

    if api_res.success:
        object_exists = True

    if object_exists:
        # Object already exists, update parameters
        print(f"Updating {obj_type} object: {host_object}")
        api_res = api_client.api_call(command=f"set-{obj_type}", payload=host_object)
    else:
        print(f"Configuring {obj_type} object: {host_object}")
        api_res = api_client.api_call(command=f"add-{obj_type}", payload=host_object)

# ... 3 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.46
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/object_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.47

Source: `python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```markdown
### Firewall Policy Exercise1

### The management rules for the Ansible Server must be installed prior to this exercise.

Connect to the Mgmt API using the Chkpnt SDK. Read your credentials in using the .env file and load_dotenv().

Re-use or re-implement your function named 'cfg_host_objects' that you previously created in the "./class3/exercise/chkpnt_sdk_ex" exercises. 

Note, in my reference function I added a second argument, 'host_object'. Consequently, I now pass the 'api_client' and the 'host_object' into the function. The 'host_object' is the host dictionary I am configuring.

'''python
def cfg_host_object(api_client, host_object):
'''

This function should create the following host object:

'''python
corp_web_server = {
    "name": "Corp Web Server",
    "ipv4-address": "172.31.144.220",
    "color": "dark green",
}
'''

# ... 48 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.47
# Source: python_course_mar26/class3/exercises/fw_policy_ex1/fw_policy_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.48

Source: `python_course_mar26/class3/exercises/object_func_ex/common_object_function.md`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```markdown
### Exercise on creating a common object function.

In the Check Point SDK exercises from Wednesday, we created host objects, network objects, and a group object.

This code had a common pattern of use f"show-{obj_type}" to check for the existence of an object (based on the name of the object).

If the object exists, then use f"set-{obj_type}" to update the existing object.

If the object doesn't exist, then use f"add-{obj_type}" to add the existing object.

Given the above, you should be able to create a common function to implement this logic. This common function would support host objects, network objects, and group objects.

Your function signature should look as follows:

'''python
def cfg_object(api_client, obj_type, obj_params):
'''

If your "set" or "add" operation fails, then you should raise the following exception.

'''python
msg = f"Failed to configure {obj_type} object: {obj_params}"
raise ChkPntConfigError(msg)
'''
# ... 25 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.48
# Source: python_course_mar26/class3/exercises/object_func_ex/common_object_function.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 17.49

Source: `python_course_mar26/class3/exercises/object_func_ex/object_function_tests.md`

**What This Example Is For**

It checks automation logic with tests, so mistakes are caught before a script touches real infrastructure.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `authenticate, request network data, paginate, and process API responses`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`API endpoint -> request/session -> JSON dicts -> object/config action`

**Try This Next**

Add one failing test first, then change the function until the test passes.

**Source Preview**

```markdown
### Test object functions

Use the referenced 'conftest.py' file as a test fixture. You will need to change the 'api_server' variable in this file to match your pod.

'''python
# Change to your pod
api_server = "chkpnt-pod99.lasthop.io"
'''

Create a 'test_object_funcs.py' file. This file should import the three object creation functions.

'''python
from object_funcs import cfg_host_object, cfg_network_object, cfg_group_object
'''

You should create the following three tests:

'''python
def test_host_object_creation(web_api_session):
'''

'''python
def test_network_object_creation(web_api_session):
'''
# ... 13 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 17.49
# Source: python_course_mar26/class3/exercises/object_func_ex/object_function_tests.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 17: Network APIs Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 18: Building a Network Automation Tool

    **Bridge from Python Crash Course**

    PCC Chapter 18 starts a web app by organizing models, views, and templates.

    **Network Automation Translation**

    1. This chapter teaches shape scripts into a small tool with clear inputs, outputs, and modules in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    operator request -> validation -> automation function -> rendered report
    ```

    **Assigned twin-bridges examples**

    - `python_course_mar26/class4/exercises/main_project/blocked_ip_funcs.py`
- `python_course_mar26/class4/exercises/main_project/run_scripts.sh`
- `python_course_mar26/class4/exercises/main_project/main_project.md`
- `python_course_mar26/class4/exercises/main_project/01_gaia_cfg_settings.py`
- `python_course_mar26/class4/exercises/main_project/02_gaia_ssh_cfg.py`
- `python_course_mar26/class4/exercises/main_project/blocked_ips.txt`
- `python_course_mar26/class4/exercises/main_project/gaia_check_password_policy.py`
- `python_course_mar26/class4/exercises/main_project/gen_fw_rules.py`
- `python_course_mar26/class4/exercises/main_project/host_objects.py`
- `python_course_mar26/class4/run_script/run_script_gaia.py`
- `python_course_mar26/class4/run_script/run_script_mgmt.py`

#### Bridge Example: Tool pipeline

In [ ]:
def validate_request(request):
    return {"vlan", "devices"}.issubset(request)


def build_plan(request):
    return [f"Configure VLAN {request['vlan']} on {device}" for device in request["devices"]]


request = {"vlan": 20, "devices": ["edge-sw1", "edge-sw2"]}
if validate_request(request):
    print(build_plan(request))

#### Bridge Example: Render a tiny report

In [ ]:
results = [{"device": "edge-sw1", "changed": True}, {"device": "edge-sw2", "changed": False}]

for result in results:
    marker = "changed" if result["changed"] else "skipped"
    print(f"{result['device']}: {marker}")

#### Twin-bridges source map 18.1

Source: `python_course_mar26/class4/exercises/main_project/blocked_ip_funcs.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `ipdb, rich.print`.
2. It defines reusable function(s): `read_blocked_ips_file, gen_host_object, get_current_blocked_ips`.
3. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import ipdb  # noqa
from rich import print  # noqa


def read_blocked_ips_file():
    # Retrieve the 'new' blocked IPs
    with open("blocked_ips.txt") as f:
        new_blocked_ips = f.readlines()
        # strip trailing newline w/ list comprehension
        new_blocked_ips = [ip.strip() for ip in new_blocked_ips]
        return new_blocked_ips


def gen_host_object(ip_addr):
    return {
        "name": ip_addr,
        "ipv4-address": ip_addr,
        "color": "black",
    }


def get_current_blocked_ips(api_client, group_name):
    """Retrieve current Blocked IPs group membership."""
    current_blocked_ips = []
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.1
# Source: python_course_mar26/class4/exercises/main_project/blocked_ip_funcs.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.2

Source: `python_course_mar26/class4/exercises/main_project/run_scripts.sh`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. This is a shell helper script.
2. It strings together command-line steps so the same setup or test can be repeated.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```bash
#!/bin/bash
python 01_gaia_cfg_settings.py
python 02_gaia_ssh_cfg.py
python 03_mgmt_api_cfg.py
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.2
# Source: python_course_mar26/class4/exercises/main_project/run_scripts.sh
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.3

Source: `python_course_mar26/class4/exercises/main_project/main_project.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
# Main Project

### Gaia Configuration using API

DNS Config (endpoint: set-dns)

'''yaml
primary: 172.31.0.2
secondary: 8.8.8.8
tertiary: 8.8.4.4
suffix: lasthop.io
'''

Static Route (endpoint: set-static-route)

'''yaml
network: 172.31.128.0/21
next_hop_gateway: 172.31.128.1
'''

### Gaia Configuration using Netmiko-SSH

'''bash
set password-controls complexity 3
# ... 114 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.3
# Source: python_course_mar26/class4/exercises/main_project/main_project.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.4

Source: `python_course_mar26/class4/exercises/main_project/01_gaia_cfg_settings.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines class blueprint(s): `GaiaConfigError`.
3. It defines reusable function(s): `config_dns, config_static_route, main`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
from rich import print  # noqa
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


class GaiaConfigError(Exception):
    pass


def config_dns(api_client):

    # DNS
    payload = {
        "primary": "172.31.0.2",
        "secondary": "8.8.8.8",
        "tertiary": "8.8.4.4",
        "suffix": "lasthop.io",
    }

    api_endpoint = "set-dns"
    api_res = api_client.api_call(command=api_endpoint, payload=payload)
    if not api_res.success:
# ... 48 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.4
# Source: python_course_mar26/class4/exercises/main_project/01_gaia_cfg_settings.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.5

Source: `python_course_mar26/class4/exercises/main_project/02_gaia_ssh_cfg.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. Then it sends configuration commands, which is the part that can change a real device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    # "session_log": "output.log",
    "secret": secret,
}

with ConnectHandler(**chkpt_fw) as ssh_conn:
    cfg_commands = [
        "set password-controls complexity 3",
        "set password-controls deny-on-nonuse enable on",
# ... 7 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.5
# Source: python_course_mar26/class4/exercises/main_project/02_gaia_ssh_cfg.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.6

Source: `python_course_mar26/class4/exercises/main_project/blocked_ips.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
10.242.1.2
10.242.1.3
10.242.1.4
10.242.1.11
10.242.1.110
10.242.1.111
10.242.1.112
10.242.1.113
10.242.1.114
10.242.1.115
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.6
# Source: python_course_mar26/class4/exercises/main_project/blocked_ips.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.7

Source: `python_course_mar26/class4/exercises/main_project/gaia_check_password_policy.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, ipdb, operator, rich.print, dotenv.load_dotenv, cpapi.APIClient`.
2. It defines reusable function(s): `condition_check, check_password_policy, check_users, main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import ipdb  # noqa
import operator
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def condition_check(cond1, cond2, comparator):
    """Function to consolidate the condition check code."""

    CHECK_PASSED = True
    operations = {
        "==": operator.eq,
        "!=": operator.ne,
        "<=": operator.le,
        ">=": operator.ge,
        "<": operator.lt,
        ">": operator.gt,
    }

    if comparator not in operations:
        raise ValueError(f"Invalid comparator: {comparator}")

# ... 115 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.7
# Source: python_course_mar26/class4/exercises/main_project/gaia_check_password_policy.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.8

Source: `python_course_mar26/class4/exercises/main_project/gen_fw_rules.py`

**What This Example Is For**

It wraps repeated network work in functions, so the same idea can be reused safely.

**Explain It Like You Are New**

1. It defines reusable function(s): `gen_blockedip_fw_rules, gen_mgmt_fw_rules`.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
def gen_blockedip_fw_rules():
    blacklisted_ips = [
        {
            "layer": "Network",
            "name": "Blacklisted IPs",
            "source": "Blocked IPs",
            "destination": "Any",
            "service": "Any",
            "action": "Drop",
            "position": 1,
        },
    ]

    return blacklisted_ips


def gen_mgmt_fw_rules(fw_name):
    management_rules = [
        {
            "layer": "Network",
            "name": "Ansible Management Access",
            "source": [
                "Ansible Server",
                "Windows SmartConsole",
# ... 19 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.8
# Source: python_course_mar26/class4/exercises/main_project/gen_fw_rules.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.9

Source: `python_course_mar26/class4/exercises/main_project/host_objects.py`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. It runs from top to bottom: create values, transform them, and print or return a result.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
smart_console_private = {
    "name": "Windows SmartConsole",
    "ipv4-address": "172.31.12.101",
    "color": "red",
}
smart_console_public = {
    "name": "Windows SmartConsole Public",
    "ipv4-address": "3.71.9.240",
    "color": "red",
}
ansible_server = {
    "name": "Ansible Server",
    "ipv4-address": "3.125.34.232",
    "color": "black",
}
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.9
# Source: python_course_mar26/class4/exercises/main_project/host_objects.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.10

Source: `python_course_mar26/class4/run_script/run_script_gaia.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, base64, rich.print, dotenv.load_dotenv, cpapi.APIClient, cpapi.APIClientArgs`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import base64
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    api_version = "1.8"
    no_ssl_verify = True

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=no_ssl_verify, context="gaia_api"
    )

    with APIClient(client_args) as api_client:
        api_client.login(username, password)
# ... 25 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.10
# Source: python_course_mar26/class4/run_script/run_script_gaia.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 18.11

Source: `python_course_mar26/class4/run_script/run_script_mgmt.py`

**What This Example Is For**

It talks to a network API, which means Python asks a controller or firewall manager for data or changes.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, base64, ipdb, rich.print, dotenv.load_dotenv, cpapi.APIClient`.
2. It defines reusable function(s): `main`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `shape scripts into a small tool with clear inputs, outputs, and modules`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator request -> validation -> automation function -> rendered report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
import os
import base64
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from cpapi import APIClient, APIClientArgs


def main():
    host = "chkpnt-pod99.lasthop.io"
    fw_name = "chkpnt-pod99"

    # This looks for a .env file and loads it
    load_dotenv()
    username = "admin"
    password = os.environ["CHKP_ADMIN"]

    api_version = "1.8"
    no_ssl_verify = True

    client_args = APIClientArgs(
        server=host, api_version=api_version, unsafe=no_ssl_verify, context="web_api"
    )

# ... 29 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 18.11
# Source: python_course_mar26/class4/run_script/run_script_mgmt.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 18: Building a Network Automation Tool Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 19: Credentials, Ownership, and Guardrails

    **Bridge from Python Crash Course**

    PCC Chapter 19 adds user accounts, protected data, forms, and ownership.

    **Network Automation Translation**

    1. This chapter teaches separate secrets, authorization, device ownership, and approval checks in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    operator identity -> permissions -> owned device set -> protected action
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class6/collateral/ssh_keys.py`
- `netmiko_course/class6/collateral/ssh_keys_agent.py`
- `netmiko_course/class6/collateral/ssh_keys_encr.py`
- `netmiko_course/class7/collateral/auth_fail.py`
- `netmiko_course/class7/collateral/auth_fail_keys.py`
- `netmiko_course/class7/collateral/auth_retry.py`
- `netmiko_course/class7/collateral/auth_retry_func.py`
- `python_course_mar26/class4/ssh_session/mgmt_cli_session.md`
- `python_course_mar26/class2/gaia_ssh/mgmt_cli_sessions.py`
- `python_course_mar26/class4/ssh_session/conftest.py`
- `python_course_mar26/class4/ssh_session/mgmt_cli_session.py`
- `netmiko_course/class6/exercises/exercise1.py`
- `netmiko_course/class6/exercises/exercise2.py`
- `netmiko_course/class6/exercises/exercise3.py`
- `netmiko_course/class6/exercises/my_ssh_config`
- `netmiko_course/class7/exercises/exercise1.py`
- `netmiko_course/class7/exercises/exercise2.py`
- `netmiko_course/class7/exercises/exercise3.py`
- `netmiko_course/class7/exercises/exercise4.py`
- `python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.md`
- `python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.py`
- `python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.md`
- `python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.py`
- `netmiko_course/class6/collateral/lab_devices.yml`
- `netmiko_course/class6/collateral/ssh_config_file.py`
- `netmiko_course/class6/collateral/ssh_notes.txt`
- `netmiko_course/class6/collateral/ssh_proxy_jump.py`
- `netmiko_course/class7/collateral/banner_fail.py`
- `netmiko_course/class7/collateral/conn_log.py`
- `netmiko_course/class7/collateral/devices.py`
- `netmiko_course/class7/collateral/dns_fail.py`
- `netmiko_course/class7/collateral/handle_failures.py`
- `netmiko_course/class7/collateral/lab_devices.yml`
- `netmiko_course/class7/collateral/tcp_conn_fail.py`
- `python_course_mar26/class2/gaia_ssh/cfg_domain_name.py`
- `python_course_mar26/class2/gaia_ssh/retrieve_fingerprint.py`
- `python_course_mar26/class2/gaia_ssh/show_version.py`
- `python_course_mar26/class4/concurrency/ssh_procs_ascompleted.py`
- `python_course_mar26/class4/concurrency/ssh_procs_ascompleted_cm.py`
- `python_course_mar26/class4/concurrency/ssh_threads_ascompleted.py`
- `python_course_mar26/class4/concurrency/ssh_threads_ascompleted_cm.py`
- `python_course_mar26/class4/concurrency/ssh_threads_wait.py`
- `python_course_mar26/work/ssh_conn_ex.py`

#### Bridge Example: Permission guard

In [ ]:
user = {"name": "alex", "roles": {"read-only", "change-requester"}}
requested_action = "configure_vlan"

if requested_action == "configure_vlan" and "network-admin" not in user["roles"]:
    print("Denied: configuration requires network-admin.")
else:
    print("Approved.")

#### Bridge Example: Secrets stay out of code

In [ ]:
import os

username = os.getenv("NETOPS_USERNAME", "demo-user")
password_loaded = bool(os.getenv("NETOPS_PASSWORD"))

print(f"Username source is environment/default: {username}")
print(f"Password loaded from environment: {password_loaded}")

#### Twin-bridges source map 19.1

Source: `netmiko_course/class6/collateral/ssh_keys.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "student1",
    "use_keys": True,
    "key_file": "~/.ssh/student_key",
    "disable_sha2_fix": True,
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.1
# Source: netmiko_course/class6/collateral/ssh_keys.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.2

Source: `netmiko_course/class6/collateral/ssh_keys_agent.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

# Key file is now encrypted
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "testuser",
    "use_keys": True,
    "key_file": "~/.ssh/test_rsa_encr",
    "allow_agent": True,
    "disable_sha2_fix": True,
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.2
# Source: netmiko_course/class6/collateral/ssh_keys_agent.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.3

Source: `netmiko_course/class6/collateral/ssh_keys_encr.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

# Key file is now encrypted
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "testuser",
    # Just make the password be the key passphrase
    "password": "cisco123",
    "use_keys": True,
    "key_file": "~/.ssh/test_rsa_encr",
    "disable_sha2_fix": True,
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.3
# Source: netmiko_course/class6/collateral/ssh_keys_encr.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.4

Source: `netmiko_course/class7/collateral/auth_fail.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": "invalid",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.4
# Source: netmiko_course/class7/collateral/auth_fail.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.5

Source: `netmiko_course/class7/collateral/auth_fail_keys.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "testuser",
    "use_keys": True,
    "key_file": "~/.ssh/id_rsa",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.5
# Source: netmiko_course/class7/collateral/auth_fail_keys.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.6

Source: `netmiko_course/class7/collateral/auth_retry.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoAuthenticationException`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler
from netmiko import NetmikoAuthenticationException


# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": "invalid",
}

output = ""
try:
    net_connect = ConnectHandler(**cisco3)
    output = net_connect.send_command("show ip arp")
except NetmikoAuthenticationException:
    print("Initial auth failed")
    cisco3["password"] = password
    net_connect = ConnectHandler(**cisco3)
    output = net_connect.send_command("show ip arp")
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.6
# Source: netmiko_course/class7/collateral/auth_retry.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.7

Source: `netmiko_course/class7/collateral/auth_retry_func.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoAuthenticationException`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `try_passwords`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler
from netmiko import NetmikoAuthenticationException


def try_passwords(device, passwords=None):
    """
    Retry using all of the passwords provided.

    passwords is an iterator of passwords to try.
    """
    if passwords is None:
        passwords = []
    for passwd in passwords:
        device["password"] = passwd
        try:
            net_connect = ConnectHandler(**device)
            break
        except NetmikoAuthenticationException:
            continue
    else:
        # nobreak
        raise NetmikoAuthenticationException("No valid password found.")
# ... 23 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.7
# Source: netmiko_course/class7/collateral/auth_retry_func.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.8

Source: `python_course_mar26/class4/ssh_session/mgmt_cli_session.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.
- Tests let you practice failure in a harmless place before a network change window.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```markdown
### Part1: Mgmt CLI Exercise

Using your pod and Netmiko SSH connect to your pod and enter expert mode.

Create a function named 'mgmt_cli_auth' with the following function signature:

'''python
def mgmt_cli_auth(ssh_conn, username, password):
'''

This function should execute the following to login to the mgmt_cli:

'''bash
cmd = f'mgmt_cli login user "{username}" password "{password}" --format json'
'''

Use Netmiko to send this command to the remote pod. Retrieve the response and process it as JSON.

From the returned data structure extract the session ID ("sid" key).

Your function should return this session ID.


### Part2: py.test testing of the 'mgmt_cli_auth' function.
# ... 10 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.8
# Source: python_course_mar26/class4/ssh_session/mgmt_cli_session.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.9

Source: `python_course_mar26/class2/gaia_ssh/mgmt_cli_sessions.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, json, sys, time, ipdb, rich.print`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import json
import sys
import time
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# mS values for various times
ONE_DAY_MS = 86_400_000
ONE_HOUR_MS = 3_600_000

# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
# ... 83 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.9
# Source: python_course_mar26/class2/gaia_ssh/mgmt_cli_sessions.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.10

Source: `python_course_mar26/class4/ssh_session/conftest.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `pytest, os, dotenv.load_dotenv, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import pytest
import os
from dotenv import load_dotenv
from netmiko import ConnectHandler


@pytest.fixture(scope="module")
def ssh_conn():

    load_dotenv()
    secret = os.environ["CHKP_EXPERT"]

    test_device = {
        "host": "chkpnt-pod99.lasthop.io",
        "device_type": "checkpoint_gaia",
        "username": "admin",
        "use_keys": True,
        "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
        "secret": secret,
    }

    with ConnectHandler(**test_device) as ssh_conn:
        ssh_conn.enable()

# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.10
# Source: python_course_mar26/class4/ssh_session/conftest.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.11

Source: `python_course_mar26/class4/ssh_session/mgmt_cli_session.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `os, json, rich.print, netmiko.ConnectHandler, dotenv.load_dotenv`.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `mgmt_cli_auth, main`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
#!/usr/bin/env python
import os
import json
from rich import print
from netmiko import ConnectHandler

from dotenv import load_dotenv


def mgmt_cli_auth(ssh_conn, username, password):
    """Use 'mgmt_cli' to authenticate and return the session_id."""

    cmd = f'mgmt_cli login user "{username}" password "{password}" --format json'
    auth_data = ssh_conn.send_command(cmd)

    auth_dict = json.loads(auth_data)
    return auth_dict["sid"]


def main():

    load_dotenv()
    secret = os.environ["CHKP_EXPERT"]
    admin_pass = os.environ["CHKP_ADMIN"]
# ... 19 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.11
# Source: python_course_mar26/class4/ssh_session/mgmt_cli_session.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.12

Source: `netmiko_course/class6/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

base_device = {
    "device_type": "cisco_ios",
    "username": "student1",
    "password": "cisco123",
    "use_keys": True,
    "key_file": "~/.ssh/student_key",
    "disable_sha2_fix": True,
}

cisco3 = base_device.copy()
cisco3["host"] = "cisco3.lasthop.io"
cisco4 = base_device.copy()
cisco4["host"] = "cisco4.lasthop.io"

for device in (cisco3, cisco4):
    with ConnectHandler(**device) as net_connect:
        print()
        print(net_connect.find_prompt())
        print("-" * 12)
        output = net_connect.send_command("show ip arp")
        print(f"{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.12
# Source: netmiko_course/class6/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.13

Source: `netmiko_course/class6/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

# Keyfile should be encrypted at this point
# note, the passphrase is not provided as we are using an SSH Agent
base_device = {
    "device_type": "cisco_ios",
    "username": "student1",
    "use_keys": True,
    "key_file": "~/.ssh/student_key",
    "allow_agent": True,
    "disable_sha2_fix": True,
}

cisco3 = base_device.copy()
cisco3["host"] = "cisco3.lasthop.io"
cisco4 = base_device.copy()
cisco4["host"] = "cisco4.lasthop.io"

for device in (cisco3, cisco4):
    with ConnectHandler(**device) as net_connect:
        print()
        print(net_connect.find_prompt())
        print("-" * 12)
        output = net_connect.send_command("show ip arp")
# ... 1 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.13
# Source: netmiko_course/class6/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.14

Source: `netmiko_course/class6/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
3. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
4. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
5. After that, it sends a show command and saves the text that comes back from the device.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
#!/usr/bin/env python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "ssh_config_file": "./my_ssh_config",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show users")

print(output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.14
# Source: netmiko_course/class6/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.15

Source: `netmiko_course/class6/exercises/my_ssh_config`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
host jumphost
  IdentitiesOnly yes
  IdentityFile ~/.ssh/my_ssh_key
  User student1
  HostName localhost

host * !jumphost
  User pyclass
  # Force usage of this SSH config file
  ProxyCommand ssh -F ~/netmiko_course/class6/exercises/my_ssh_config -W %h:%p jumphost
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.15
# Source: netmiko_course/class6/exercises/my_ssh_config
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.16

Source: `netmiko_course/class7/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoAuthenticationException`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `netmiko_connect`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, NetmikoAuthenticationException


def netmiko_connect(device):
    """
    Successful connection returns: (True, connect_obj)

    Failed authentication returns: (False, None)
    """
    try:
        net_connect = ConnectHandler(**device)
        return (True, net_connect)
    except NetmikoAuthenticationException:
        return (False, None)


if __name__ == "__main__":

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )
# ... 20 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.16
# Source: netmiko_course/class7/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.17

Source: `netmiko_course/class7/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoTimeoutException, netmiko.NetmikoAuthenticationException`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `netmiko_connect`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler
from netmiko import NetmikoTimeoutException, NetmikoAuthenticationException


def netmiko_connect(device):
    """
    Successful connection returns: (True, connect_obj)

    Failed authentication returns: (False, None)
    """
    try:
        net_connect = ConnectHandler(**device)
        return (True, net_connect)
    except NetmikoAuthenticationException:
        print("\nAuthentication failed")
        return (False, None)
    except NetmikoTimeoutException:
        print("\nConnection failed")
        return (False, None)


if __name__ == "__main__":
# ... 27 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.17
# Source: netmiko_course/class7/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.18

Source: `netmiko_course/class7/exercises/exercise3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoTimeoutException, netmiko.NetmikoAuthenticationException, logging`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It defines reusable function(s): `netmiko_connect`.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler
from netmiko import NetmikoTimeoutException, NetmikoAuthenticationException

import logging

logging.basicConfig(
    filename="netmiko_class7.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)
logger = logging.getLogger(__name__)


def netmiko_connect(device_name, device):
    """
    Successful connection returns: (True, connect_obj)

    Failed authentication returns: (False, None)
    """
    hostname = device["host"]
    port = device.get("port", 22)
    msg = ""
# ... 77 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.18
# Source: netmiko_course/class7/exercises/exercise3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.19

Source: `netmiko_course/class7/exercises/exercise4.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnLogOnly`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnLogOnly


if __name__ == "__main__":

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    # DNS failure
    vmx1 = {
        "name": "vmx1",
        "device_type": "juniper_junos",
        "host": "invalid.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

    # Invalid Port
    vmx2 = {
        "name": "vmx2",
# ... 33 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.19
# Source: netmiko_course/class7/exercises/exercise4.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.20

Source: `python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
### Gaia SSH Exercise1

Connect to your lab pod using Netmiko. You will need to use an SSH key to connect. This connection will require the following Netmiko arguments:

'''python
chkpt_fw = {
    "host": "chkpnt-podN.lasthop.io",       # REPLACE with your pod
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,       # NEEDED for SSH Key
    "key_file": "/home/studentN/.ssh/eu-sshkey.pem",    # REPLACE with your student
}
'''

Using this SSH connection, execute "show arp dynamic all" and print this response out to standard output.

Your output should look similar to the following:

'''bash
$ python gaia_ssh_ex1.py 
Dynamic Arp Parameters

IP Address                 Mac Address                
172.31.32.1             0a:61:33:92:44:55
# ... 5 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.20
# Source: python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.21

Source: `python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `rich.print, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from rich import print
from netmiko import ConnectHandler

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "session_log": "output.log",
}

with ConnectHandler(**chkpt_fw) as nc:

    cmd = "show arp dynamic all"
    data = nc.send_command(cmd)
    print(data)
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.21
# Source: python_course_mar26/class2/exercises/gaia_ssh_ex/gaia_ssh_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.22

Source: `python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```markdown
### mgmt_cli SSH Exercise1

Connect to your lab pod using Netmiko. You will once again need to use an SSH key to connect.

You will also need to provide the Check Point "expert" credentials. I recommend you do this by using .env file and the following pattern.

'''python
from dotenv import load_dotenv

# This looks for a .env file and loads it
load_dotenv()
secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]
'''

You will then use this 'secret' variable in your Netmiko ConnectHandler arguments.

Once connected, call the .enable() method to elevate privileges (this should cause you to enter 'expert' mode).

Now execute the following:

'''python
cmd = f'''mgmt_cli login user "admin" password "{admin_pass}" --format json'''
data = ssh_conn.send_command(cmd)
# ... 36 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.22
# Source: python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.23

Source: `python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, json, time, ipdb, rich.print, dotenv.load_dotenv`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import json
import time
import ipdb  # noqa
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# This looks for a .env file and loads it
load_dotenv()
secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "session_log": "output.log",
    "secret": secret,
}

with ConnectHandler(**chkpt_fw) as ssh_conn:
# ... 28 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.23
# Source: python_course_mar26/class2/exercises/gaia_ssh_ex/mgmt_cli_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.24

Source: `netmiko_course/class6/collateral/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
# Dictionaries are devices
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
# ... 46 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.24
# Source: netmiko_course/class6/collateral/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.25

Source: `netmiko_course/class6/collateral/ssh_config_file.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
3. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
4. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
5. After that, it sends a show command and saves the text that comes back from the device.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
#!/usr/bin/env python
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "ssh_config_file": "~/.ssh/ssh_config",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show users")

print(output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.25
# Source: netmiko_course/class6/collateral/ssh_config_file.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.26

Source: `netmiko_course/class6/collateral/ssh_notes.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
# Encrypt an existing SSH key
ssh-keygen -o -p -f ./test_rsa

# Run SSH Agent
ssh-agent 

# Obviously, with the right values 
SSH_AUTH_SOCK=/tmp/ssh-7PGNJNwQ9OgN/agent.30115; export SSH_AUTH_SOCK;
SSH_AGENT_PID=30116; export SSH_AGENT_PID;

# Add keys to SSH Agent
ssh-add ~/.ssh/test_rsa
ssh-add -l

# Paramiko will automatically look for:
# Any “id_rsa”, “id_dsa” or “id_ecdsa” key discoverable in ~/.ssh/
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.26
# Source: netmiko_course/class6/collateral/ssh_notes.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.27

Source: `netmiko_course/class6/collateral/ssh_proxy_jump.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `os, netmiko.ConnectHandler, getpass.getpass`.
3. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
4. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
5. After that, it sends a show command and saves the text that comes back from the device.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
#!/usr/bin/env python
import os
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "ssh_config_file": "/home/kbyers/.ssh/ssh_config_proxyjump",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show users")

print(output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.27
# Source: netmiko_course/class6/collateral/ssh_proxy_jump.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.28

Source: `netmiko_course/class7/collateral/banner_fail.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "testuser",
    "use_keys": True,
    "key_file": "~/.ssh/test_rsa",
    "banner_timeout": 1,
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.28
# Source: netmiko_course/class7/collateral/banner_fail.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.29

Source: `netmiko_course/class7/collateral/conn_log.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `logging, netmiko.ConnLogOnly, devices.cisco3, devices.cisco4, devices.arista1, devices.arista2`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import logging
from netmiko import ConnLogOnly
from devices import cisco3, cisco4, arista1, arista2


log_level = logging.INFO
log_file = "my_output.log"

for device in (cisco3, cisco4, arista1, arista2):
    net_connect = ConnLogOnly(log_file=log_file, log_level=log_level, **device)
    if net_connect:
        print(net_connect.find_prompt())
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.29
# Source: netmiko_course/class7/collateral/conn_log.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.30

Source: `netmiko_course/class7/collateral/devices.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
import os
from getpass import getpass

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_xe",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": "invalid",
}

cisco4 = {
    "device_type": "cisco_xe",
    "host": "cisco4.lasthop.io",
    "username": "pyclass",
    "password": password,
}

arista1 = {
    "device_type": "arista_eos",
    "host": "arista1.lasthop.io",
    "username": "pyclass",
# ... 10 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.30
# Source: netmiko_course/class7/collateral/devices.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.31

Source: `netmiko_course/class7/collateral/dns_fail.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "invalid.lasthop.io",
    "username": "testuser",
    "use_keys": True,
    "key_file": "~/.ssh/test_rsa",
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.31
# Source: netmiko_course/class7/collateral/dns_fail.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.32

Source: `netmiko_course/class7/collateral/handle_failures.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, yaml, getpass.getpass, netmiko.ConnectHandler, netmiko.NetmikoAuthenticationException, netmiko.NetmikoTimeoutException`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. It defines reusable function(s): `load_devices, netmiko_conn`.
5. It reads or writes files, which is how automation remembers inventory, commands, or reports.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import yaml
from getpass import getpass
from netmiko import ConnectHandler
from netmiko import NetmikoAuthenticationException
from netmiko import NetmikoTimeoutException
from paramiko.ssh_exception import SSHException


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


def netmiko_conn(device):
    try:
        conn = ConnectHandler(**device)
        return conn
    except NetmikoTimeoutException as e:
        if "DNS failure" in str(e):
            print("DNS failure")
        elif "TCP connection to device failed" in str(e):
# ... 28 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.32
# Source: netmiko_course/class7/collateral/handle_failures.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.33

Source: `netmiko_course/class7/collateral/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
cisco3:
  device_type: cisco_xe
  host: invalid.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass
  port: 8022

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass
  password: invalid

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
# ... 28 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.33
# Source: netmiko_course/class7/collateral/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.34

Source: `netmiko_course/class7/collateral/tcp_conn_fail.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "testuser",
    "use_keys": True,
    "key_file": "~/.ssh/test_rsa",
    "port": 8022,
    "conn_timeout": 8,
}

with ConnectHandler(**cisco3) as net_connect:
    output = net_connect.send_command("show ip arp")

print(f"\n{output}\n")
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.34
# Source: netmiko_course/class7/collateral/tcp_conn_fail.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.35

Source: `python_course_mar26/class2/gaia_ssh/cfg_domain_name.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. Then it sends configuration commands, which is the part that can change a real device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "session_log": "output.log",
    "secret": secret,
}

with ConnectHandler(**chkpt_fw) as ssh_conn:

    print(ssh_conn.find_prompt())

# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.35
# Source: python_course_mar26/class2/gaia_ssh/cfg_domain_name.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.36

Source: `python_course_mar26/class2/gaia_ssh/retrieve_fingerprint.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, rich.print, dotenv.load_dotenv, netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from rich import print
from dotenv import load_dotenv
from netmiko import ConnectHandler

# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    # "session_log": "output.log",
    "secret": secret,
}

with ConnectHandler(**chkpt_fw) as ssh_conn:
    print(ssh_conn.find_prompt())

# ... 13 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.36
# Source: python_course_mar26/class2/gaia_ssh/retrieve_fingerprint.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.37

Source: `python_course_mar26/class2/gaia_ssh/show_version.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler

chkpt_fw = {
    "host": "chkpnt-pod99.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "session_log": "output.log",
}

with ConnectHandler(**chkpt_fw) as nc:
    print(nc.find_prompt())

    cmd = "show version all"
    data = nc.send_command(cmd)
    print(data)
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.37
# Source: python_course_mar26/class2/gaia_ssh/show_version.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.38

Source: `python_course_mar26/class4/concurrency/ssh_procs_ascompleted.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ProcessPoolExecutor, concurrent.futures.as_completed, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    return net_connect.find_prompt()


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 4

    pool = ProcessPoolExecutor(max_threads)

    future_list = []
    for a_device in device_list:
        future = pool.submit(ssh_conn, a_device)
        future_list.append(future)

    # Process as completed
    for future in as_completed(future_list):
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.38
# Source: python_course_mar26/class4/concurrency/ssh_procs_ascompleted.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.39

Source: `python_course_mar26/class4/concurrency/ssh_procs_ascompleted_cm.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ProcessPoolExecutor, concurrent.futures.as_completed, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    return net_connect.find_prompt()


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 4

    # Use context manager to gracefully cleanup the pool
    with ProcessPoolExecutor(max_threads) as pool:
        future_list = []
        for a_device in device_list:
            future = pool.submit(ssh_conn, a_device)
            future_list.append(future)

        # Process as completed
        for future in as_completed(future_list):
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.39
# Source: python_course_mar26/class4/concurrency/ssh_procs_ascompleted_cm.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.40

Source: `python_course_mar26/class4/concurrency/ssh_threads_ascompleted.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    return net_connect.find_prompt()


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 4

    pool = ThreadPoolExecutor(max_threads)

    future_list = []
    for a_device in device_list:
        future = pool.submit(ssh_conn, a_device)
        future_list.append(future)

    # Process as completed
    for future in as_completed(future_list):
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.40
# Source: python_course_mar26/class4/concurrency/ssh_threads_ascompleted.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.41

Source: `python_course_mar26/class4/concurrency/ssh_threads_ascompleted_cm.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    return net_connect.find_prompt()


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 4

    # Use context manager to gracefully cleanup the pool
    with ThreadPoolExecutor(max_threads) as pool:
        future_list = []
        for a_device in device_list:
            future = pool.submit(ssh_conn, a_device)
            future_list.append(future)

        # Process as completed
        for future in as_completed(future_list):
# ... 4 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.41
# Source: python_course_mar26/class4/concurrency/ssh_threads_ascompleted_cm.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.42

Source: `python_course_mar26/class4/concurrency/ssh_threads_wait.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ThreadPoolExecutor, concurrent.futures.wait, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. It defines reusable function(s): `ssh_conn`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from concurrent.futures import ThreadPoolExecutor, wait
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    return net_connect.find_prompt()


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 4

    pool = ThreadPoolExecutor(max_threads)

    future_list = []
    for a_device in device_list:
        future = pool.submit(ssh_conn, a_device)
        future_list.append(future)

    # Waits until all the pending threads are done
    wait(future_list)
# ... 6 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.42
# Source: python_course_mar26/class4/concurrency/ssh_threads_wait.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 19.43

Source: `python_course_mar26/work/ssh_conn_ex.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler, rich.print`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It defines reusable function(s): `main`.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `separate secrets, authorization, device ownership, and approval checks`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`operator identity -> permissions -> owned device set -> protected action`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler
from rich import print


def main():

    pod1 = {
        "host": "chkpnt-pod1.lasthop.io",
        "device_type": "checkpoint_gaia",
        "username": "admin",
        "use_keys": True,
        "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
        "session_log": "output.log",
    }
    pod99 = {
        "host": "chkpnt-pod99.lasthop.io",
        "device_type": "checkpoint_gaia",
        "username": "admin",
        "use_keys": True,
        "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
        "session_log": "output.log",
    }

    for device in (pod1, pod99):
# ... 25 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 19.43
# Source: python_course_mar26/work/ssh_conn_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 19: Credentials, Ownership, and Guardrails Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Chapter 20: Operating and Deploying Automation

    **Bridge from Python Crash Course**

    PCC Chapter 20 finishes a project by styling, configuring, deploying, and operating it.

    **Network Automation Translation**

    1. This chapter teaches package scripts, run checks, log sessions, and prepare production workflows in the language of network automation.
    2. Start with a small safe example, observe the output, change one thing, then connect the pattern to a real network task.
    3. Real device operations are shown as source-map study blocks; runnable cells use simulated data unless the cell clearly says otherwise.
    4. When you move this into a lab, replace fake inventory and outputs first, then add credentials, connectivity, and configuration writes last.

    **Concept Flow**

    ```text
    local script -> config/logging/tests -> scheduled run -> operational report
    ```

    **Assigned twin-bridges examples**

    - `netmiko_course/class9/collateral/get_file.py`
- `netmiko_course/class9/collateral/put_file.py`
- `netmiko_course/deploy.sh`
- `netmiko_course/tests.sh`
- `netmiko_course/class9/exercises/exercise1.py`
- `netmiko_course/class9/exercises/exercise2.py`
- `netmiko_course/class9/collateral/get_progress_bar.py`
- `netmiko_course/class9/collateral/put_progress_bar.py`
- `netmiko_course/class11/exercises/exercise1_final.py`
- `netmiko_course/class11/exercises/exercise1a.py`
- `netmiko_course/class11/exercises/exercise1b.py`
- `netmiko_course/class11/exercises/exercise1c.py`
- `netmiko_course/class8/exercises/exercise1.py`
- `netmiko_course/class8/exercises/exercise2.py`
- `netmiko_course/class8/exercises/lab_devices.yml`
- `netmiko_course/class8/exercises/utilities.py`
- `netmiko_course/class9/exercises/cfg_name_servers.txt`
- `netmiko_course/class9/exercises/show_interfaces_ktb.txt`
- `python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.md`
- `python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.py`
- `python_course_mar26/class4/exercises/concurrency_ex/my_devices.py`
- `netmiko_course/class11/collateral/redispatch1.py`
- `netmiko_course/class11/collateral/term_server1.py`
- `netmiko_course/class11/collateral/term_server2.py`
- `netmiko_course/class11/collateral/term_server3.py`
- `netmiko_course/class11/collateral/term_server4.py`
- `netmiko_course/class8/collateral/cf_processes.py`
- `netmiko_course/class8/collateral/cf_threads.py`
- `netmiko_course/class8/collateral/cf_threads_asc.py`
- `netmiko_course/class8/collateral/lab_devices.yml`
- `netmiko_course/class9/collateral/test.txt`
- `netmiko_course/class9/collateral/test2.txt`
- `netmiko_course/class9/collateral/testx.txt`
- `netmiko_course/.netmiko.yml`
- `netmiko_course/README.md`
- `netmiko_course/lab_devices.yml`
- `netmiko_course/requirements.txt`
- `netmiko_course/setup.cfg`
- `python_course_mar26/class4/concurrency/my_devices.py`
- `python_course_mar26/class1/exercises/func_ex/func_ex1.md`
- `python_course_mar26/class1/exercises/func_ex/func_ex1.py`
- `python_course_mar26/class1/exercises/func_ex/func_ex2.md`
- `python_course_mar26/class1/exercises/func_ex/func_ex2.py`
- `python_course_mar26/class1/exercises/func_ex/func_ex3.py`
- `python_course_mar26/class1/exercises/list_ex/locations.yml`
- `python_course_mar26/class3/exercises/fw_policy_ex1/chkpt_exceptions.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/chkpt_exceptions.py`
- `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.md`
- `python_course_mar26/README.md`
- `python_course_mar26/class1/libraries/lib_test1.py`
- `python_course_mar26/class1/libraries/lib_test2.py`
- `python_course_mar26/class1/libraries/sys_path_ex.py`
- `python_course_mar26/class3/sets/set_ex.py`
- `python_course_mar26/lib_class4/chkpt_exceptions.py`
- `python_course_mar26/work/notes.txt`

#### Bridge Example: Operational run summary

In [ ]:
run = {
    "job": "nightly-show-version",
    "devices": 12,
    "success": 11,
    "failed": ["edge-sw9"],
}

print(f"{run['job']}: {run['success']} of {run['devices']} succeeded")
if run["failed"]:
    print(f"Review failures: {', '.join(run['failed'])}")

#### Bridge Example: Release checklist

In [ ]:
checklist = [
    "unit tests pass",
    "dry-run reviewed",
    "rollback documented",
    "session logging enabled",
    "credentials loaded from environment",
]

for item in checklist:
    print(f"[ ] {item}")

#### Twin-bridges source map 20.1

Source: `netmiko_course/class9/collateral/get_file.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# Need a privilege15 account (no enable call)
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

# Secure copy server must be enable on the device ('ip scp server enable')
source_file = "test2.txt"
dest_file = "test2.txt"
direction = "get"
file_system = "flash:"

ssh_conn = ConnectHandler(**cisco3)
transfer_dict = file_transfer(
    ssh_conn,
# ... 10 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.1
# Source: netmiko_course/class9/collateral/get_file.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.2

Source: `netmiko_course/class9/collateral/put_file.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# Need a privilege15 account (no enable call)
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

# Secure copy server must be enable on the device ('ip scp server enable')
source_file = "test2.txt"
dest_file = "test2.txt"
direction = "put"
file_system = "flash:"

ssh_conn = ConnectHandler(**cisco3)
transfer_dict = file_transfer(
    ssh_conn,
# ... 9 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.2
# Source: netmiko_course/class9/collateral/put_file.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.3

Source: `netmiko_course/deploy.sh`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. This is a shell helper script.
2. It strings together command-line steps so the same setup or test can be repeated.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```bash
#mkdir class2/collateral
#mkdir class2/exercises
#touch class2/collateral/empty.txt
#touch class2/exercises/empty.txt

mkdir class3/collateral
mkdir class3/exercises
touch class3/collateral/empty.txt
touch class3/exercises/empty.txt

mkdir class4/collateral
mkdir class4/exercises
touch class4/collateral/empty.txt
touch class4/exercises/empty.txt

mkdir class5/collateral
mkdir class5/exercises
touch class5/collateral/empty.txt
touch class5/exercises/empty.txt

mkdir class6/collateral
mkdir class6/exercises
touch class6/collateral/empty.txt
touch class6/exercises/empty.txt
# ... 15 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.3
# Source: netmiko_course/deploy.sh
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.4

Source: `netmiko_course/tests.sh`

**What This Example Is For**

It checks automation logic with tests, so mistakes are caught before a script touches real infrastructure.

**Explain It Like You Are New**

1. This is a shell helper script.
2. It strings together command-line steps so the same setup or test can be repeated.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```bash
#!/bin/bash

RETURN_CODE=0

echo "pylama ." \
&& pylama . \
&& echo "black" \
&& black --check . \
&& echo "running pytest..." \
&& cd tests \
&& py.test -x -s -v test_class1.py \
&& py.test -x -s -v test_class2.py \
&& py.test -x -s -v test_class3.py \
&& py.test -x -s -v test_class4.py \
&& py.test -x -s -v test_class5.py \
&& py.test -x -s -v test_class6.py \
&& py.test -x -s -v test_class7.py \
&& py.test -x -s -v test_class8.py \
&& py.test -x -s -v test_class9.py \
&& py.test -x -s -v test_class10.py \
&& py.test -x -s -v test_class11.py \
&& py.test -x -s -v test_class12.py \
\
|| RETURN_CODE=1
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.4
# Source: netmiko_course/tests.sh
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.5

Source: `netmiko_course/class9/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Finally, it disconnects so the network session is not left hanging open.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# Need a privilege15 account (no enable call)
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}
cisco4 = {
    "device_type": "cisco_ios",
    "host": "cisco4.lasthop.io",
    "username": "pyclass",
    "password": password,
}

# Secure copy server must be enable on the device ('ip scp server enable')
source_file = "cfg_name_servers.txt"
dest_file = "cfg_name_servers.txt"
# ... 43 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.5
# Source: netmiko_course/class9/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.6

Source: `netmiko_course/class9/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `redirect_output, more_file`.
6. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
    "session_log": "output.txt",
}


def redirect_output(ssh_conn, cmd):
    print("\n\nRedirecting 'show interfaces' output to flash:")
    output = ssh_conn.send_command_timing(cmd, strip_prompt=False, strip_command=False)

    # Command will prompt for confirmation if the file already exists
    if "confirm" in output:
        output += ssh_conn.send_command_timing(
            "y", strip_prompt=False, strip_command=False
# ... 49 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.6
# Source: netmiko_course/class9/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.7

Source: `netmiko_course/class9/collateral/get_progress_bar.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer, netmiko.progress_bar`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer, progress_bar

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# Need a privilege15 account (no enable call)
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

# Secure copy server must be enable on the device ('ip scp server enable')
source_file = "testx.txt"
dest_file = "testx.txt"
direction = "get"
file_system = "flash:"

ssh_conn = ConnectHandler(**cisco3)
transfer_dict = file_transfer(
    ssh_conn,
# ... 9 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.7
# Source: netmiko_course/class9/collateral/get_progress_bar.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.8

Source: `netmiko_course/class9/collateral/put_progress_bar.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.file_transfer, netmiko.progress_bar`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, file_transfer, progress_bar

# Code so automated tests will run properly
password = os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()

# Need a privilege15 account (no enable call)
cisco3 = {
    "device_type": "cisco_ios",
    "host": "cisco3.lasthop.io",
    "username": "pyclass",
    "password": password,
}

# Secure copy server must be enable on the device ('ip scp server enable')
source_file = "testx.txt"
dest_file = "testx.txt"
direction = "put"
file_system = "flash:"

ssh_conn = ConnectHandler(**cisco3)
transfer_dict = file_transfer(
    ssh_conn,
# ... 9 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.8
# Source: netmiko_course/class9/collateral/put_progress_bar.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.9

Source: `netmiko_course/class11/exercises/exercise1_final.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.redispatch`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, redispatch


if __name__ == "__main__":

    debug = True
    arista4_internal_ip = "10.220.88.31"
    ssh_cmd = f"ssh -l pyclass {arista4_internal_ip}"

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    cisco3 = {
        "device_type": "cisco_ios",
        "host": "cisco3.lasthop.io",
        "username": "pyclass",
        "password": password,
        "session_log": "output.txt",
    }

# ... 55 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.9
# Source: netmiko_course/class11/exercises/exercise1_final.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.10

Source: `netmiko_course/class11/exercises/exercise1a.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.redispatch`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, redispatch


if __name__ == "__main__":

    debug = True
    arista4_internal_ip = "10.220.88.31"
    ssh_cmd = f"ssh -l pyclass {arista4_internal_ip}"

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    cisco3 = {
        "device_type": "cisco_ios",
        "host": "cisco3.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

    # Connect to Cisco3 using Netmiko
# ... 29 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.10
# Source: netmiko_course/class11/exercises/exercise1a.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.11

Source: `netmiko_course/class11/exercises/exercise1b.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.redispatch`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, redispatch


if __name__ == "__main__":

    debug = True
    arista4_internal_ip = "10.220.88.31"
    ssh_cmd = f"ssh -l pyclass {arista4_internal_ip}"

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    cisco3 = {
        "device_type": "cisco_ios",
        "host": "cisco3.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

    # Connect to Cisco3 using Netmiko
# ... 40 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.11
# Source: netmiko_course/class11/exercises/exercise1b.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.12

Source: `netmiko_course/class11/exercises/exercise1c.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler, netmiko.redispatch`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
from getpass import getpass
from netmiko import ConnectHandler, redispatch


if __name__ == "__main__":

    debug = True
    arista4_internal_ip = "10.220.88.31"
    ssh_cmd = f"ssh -l pyclass {arista4_internal_ip}"

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    cisco3 = {
        "device_type": "cisco_ios",
        "host": "cisco3.lasthop.io",
        "username": "pyclass",
        "password": password,
    }

    # Connect to Cisco3 using Netmiko
# ... 54 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.12
# Source: netmiko_course/class11/exercises/exercise1c.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.13

Source: `netmiko_course/class8/exercises/exercise1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, concurrent.futures.ThreadPoolExecutor, concurrent.futures.wait, getpass.getpass, utilities.load_devices, utilities.ssh_conn`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from concurrent.futures import ThreadPoolExecutor, wait
from getpass import getpass

# Store certain functions in another module so the can be used across multiple exercises.
from utilities import load_devices, ssh_conn


if __name__ == "__main__":

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    my_devices = load_devices()
    device_list = my_devices["all"]
    pool = ThreadPoolExecutor(20)

    cmd_dict = {"cisco_nxos": "show ip arp vrf management", "juniper_junos": "show arp"}

    future_list = []
    for device_name in device_list:
        device_dict = my_devices[device_name]
# ... 21 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.13
# Source: netmiko_course/class8/exercises/exercise1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.14

Source: `netmiko_course/class8/exercises/exercise2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, concurrent.futures.ProcessPoolExecutor, concurrent.futures.as_completed, getpass.getpass, utilities.load_devices, utilities.ssh_conn`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from getpass import getpass

# Store certain functions in another module so the can be used across multiple exercises.
from utilities import load_devices, ssh_conn


if __name__ == "__main__":

    # Code so automated tests will run properly
    password = (
        os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
    )

    my_devices = load_devices()
    device_list = my_devices["all"]
    pool = ProcessPoolExecutor(20)

    cmd_dict = {"cisco_nxos": "show ip arp vrf management", "juniper_junos": "show arp"}

    future_list = []
    for device_name in device_list:
        device_dict = my_devices[device_name]
# ... 21 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.14
# Source: netmiko_course/class8/exercises/exercise2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.15

Source: `netmiko_course/class8/exercises/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
  host: arista3.lasthop.io
# ... 58 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.15
# Source: netmiko_course/class8/exercises/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.16

Source: `netmiko_course/class8/exercises/utilities.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `netmiko.ConnectHandler, yaml`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It defines reusable function(s): `load_devices, ssh_conn`.
5. It reads or writes files, which is how automation remembers inventory, commands, or reports.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
from netmiko import ConnectHandler
import yaml


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


def ssh_conn(device_name, device_dict, cmd=None):
    with ConnectHandler(**device_dict) as net_connect:
        if cmd is None:
            return net_connect.find_prompt()
        else:
            output = net_connect.send_command(cmd)
            return (device_name, output)
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.16
# Source: netmiko_course/class8/exercises/utilities.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.17

Source: `netmiko_course/class9/exercises/cfg_name_servers.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
no ip name-server 1.1.1.1
no ip name-server 1.0.0.1
ip name-server 8.8.8.8
ip name-server 8.8.4.4
ip domain name lasthop.io
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.17
# Source: netmiko_course/class9/exercises/cfg_name_servers.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.18

Source: `netmiko_course/class9/exercises/show_interfaces_ktb.txt`

**What This Example Is For**

It repeats a network task over items such as devices, interfaces, commands, or retries.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```text
GigabitEthernet0/0/0 is up, line protocol is up 
  Hardware is C1111-2x1GE, address is a093.5141.b780 (bia a093.5141.b780)
  Internet address is 10.220.88.22/24
  MTU 1500 bytes, BW 100000 Kbit/sec, DLY 100 usec, 
     reliability 255/255, txload 1/255, rxload 1/255
  Encapsulation ARPA, loopback not set
  Keepalive not supported 
  Full Duplex, 100Mbps, link type is auto, media type is RJ45
  output flow-control is off, input flow-control is off
  ARP type: ARPA, ARP Timeout 04:00:00
  Last input 00:00:00, output 00:00:02, output hang never
  Last clearing of "show interface" counters never
  Input queue: 0/375/0/0 (size/max/drops/flushes); Total output drops: 0
  Queueing strategy: fifo
  Output queue: 0/40 (size/max)
  5 minute input rate 4000 bits/sec, 5 packets/sec
  5 minute output rate 6000 bits/sec, 6 packets/sec
     12503976 packets input, 1549063563 bytes, 0 no buffer
     Received 77240 broadcasts (0 IP multicasts)
     0 runts, 0 giants, 0 throttles 
     0 input errors, 0 CRC, 0 frame, 0 overrun, 0 ignored
     0 watchdog, 9931939 multicast, 0 pause input
     4837937 packets output, 1541010289 bytes, 0 underruns
     Output 68 broadcasts (0 IP multicasts)
# ... 198 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.18
# Source: netmiko_course/class9/exercises/show_interfaces_ktb.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.19

Source: `python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
### Concurrency Exercise

Connect using a thread pool and Netmiko to pods1-5. Your thread pool size should be set to 5.

Using threads execute "show interface eth0" and retreive the output. You should also record your hostname in the thread result (so you know which device the result came from). 

Use the "as_completed" pattern to print out the results as they are returned. However, only display the device hostname and the "ipv4-address" from the "show interface" output.

Record and print the total execution time for your script. 

Change the thread pool size to 2 and see how it changes your execution time.

Optional: Convert from threads over to processes. Use a process pool size of 5 and compare the execution time (threads to processes).
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.19
# Source: python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.20

Source: `python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed, datetime.datetime, netmiko.ConnectHandler, my_devices.device_list`.
2. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
3. After that, it sends a show command and saves the text that comes back from the device.
4. It defines reusable function(s): `ssh_conn`.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
# Pool size 5: 0:00:02.135320 (Threads)
# Pool size 2: 0:00:06.356690 (Threads)
#
# Pool size 5: 0:00:02.155889 (Processes)
from concurrent.futures import ThreadPoolExecutor, as_completed

# from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from netmiko import ConnectHandler
from my_devices import device_list


def ssh_conn(device):
    net_connect = ConnectHandler(**device)
    host = net_connect.host
    cmd = "show interface eth0"
    data = net_connect.send_command(cmd)
    return (host, data)


if __name__ == "__main__":
    start_time = datetime.now()
    max_threads = 5

# ... 23 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.20
# Source: python_course_mar26/class4/exercises/concurrency_ex/concurrency_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.21

Source: `python_course_mar26/class4/exercises/concurrency_ex/my_devices.py`

**What This Example Is For**

It repeats a network task over items such as devices, interfaces, commands, or retries.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, dotenv.load_dotenv`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from dotenv import load_dotenv


# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw1 = {
    "host": "chkpnt-pod1.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "secret": secret,
}
chkpt_fw2 = {
    "host": "chkpnt-pod2.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.21
# Source: python_course_mar26/class4/exercises/concurrency_ex/my_devices.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.22

Source: `netmiko_course/class11/collateral/redispatch1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, netmiko.ConnectHandler, netmiko.redispatch, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. Then it sends configuration commands, which is the part that can change a real device.
6. Finally, it disconnects so the network session is not left hanging open.
7. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Configuration commands need guardrails: dry run, approval, verification, and rollback.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Note: Students won't be able to run this code since the terminal server is not accessible.
"""
import os
import time
from netmiko import ConnectHandler, redispatch
from getpass import getpass

# Code so automated tests will run properly
password = (
    os.getenv("TERM_SERVER_PASSWORD")
    if os.getenv("TERM_SERVER_PASSWORD")
    else getpass()
)

# Code so automated tests will run properly
end_device_pwd = (
    os.getenv("NETMIKO_PASSWORD") if os.getenv("NETMIKO_PASSWORD") else getpass()
)

term_server = {
    "device_type": "generic_termserver_telnet",
    "host": "184.105.247.69",
    "username": "admin",
# ... 43 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.22
# Source: netmiko_course/class11/collateral/redispatch1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.23

Source: `netmiko_course/class11/collateral/term_server1.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, getpass.getpass, netmiko.ConnectHandler`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Note: Students won't be able to run this code since the terminal server is not accessible.
"""
import os
from getpass import getpass
from netmiko import ConnectHandler

# Code so automated tests will run properly
password = (
    os.getenv("TERM_SERVER_PASSWORD")
    if os.getenv("TERM_SERVER_PASSWORD")
    else getpass()
)

term_server = {
    "device_type": "generic_termserver_telnet",
    "host": "184.105.247.69",
    "username": "admin",
    "password": password,
    "session_log": "term_server.out",
}

net_connect = ConnectHandler(**term_server)
net_connect.std_login()
# ... 2 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.23
# Source: netmiko_course/class11/collateral/term_server1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.24

Source: `netmiko_course/class11/collateral/term_server2.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Note: Students won't be able to run this code since the terminal server is not accessible.
"""
import os
import time
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = (
    os.getenv("TERM_SERVER_PASSWORD")
    if os.getenv("TERM_SERVER_PASSWORD")
    else getpass()
)

term_server = {
    "device_type": "generic_termserver_telnet",
    "host": "184.105.247.69",
    "username": "admin",
    "password": password,
    "session_log": "term_server.out",
}

net_connect = ConnectHandler(**term_server)
# ... 15 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.24
# Source: netmiko_course/class11/collateral/term_server2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.25

Source: `netmiko_course/class11/collateral/term_server3.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Note: Students won't be able to run this code since the terminal server is not accessible.
"""
import os
import time
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = (
    os.getenv("TERM_SERVER_PASSWORD")
    if os.getenv("TERM_SERVER_PASSWORD")
    else getpass()
)

term_server = {
    "device_type": "generic_termserver_telnet",
    "host": "184.105.247.69",
    "username": "admin",
    "password": password,
    "session_log": "term_server.out",
}

net_connect = ConnectHandler(**term_server)
# ... 17 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.25
# Source: netmiko_course/class11/collateral/term_server3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.26

Source: `netmiko_course/class11/collateral/term_server4.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, time, netmiko.ConnectHandler, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. Finally, it disconnects so the network session is not left hanging open.
5. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
"""
Note: Students won't be able to run this code since the terminal server is not accessible.
"""
import os
import time
from netmiko import ConnectHandler
from getpass import getpass

# Code so automated tests will run properly
password = (
    os.getenv("TERM_SERVER_PASSWORD")
    if os.getenv("TERM_SERVER_PASSWORD")
    else getpass()
)

term_server = {
    "device_type": "generic_termserver_telnet",
    "host": "184.105.247.69",
    "username": "admin",
    "password": password,
    "session_log": "term_server.out",
}

net_connect = ConnectHandler(**term_server)
# ... 30 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.26
# Source: netmiko_course/class11/collateral/term_server4.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.27

Source: `netmiko_course/class8/collateral/cf_processes.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, yaml, concurrent.futures.ProcessPoolExecutor, concurrent.futures.wait, datetime.datetime, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `load_devices, ssh_conn`.
6. It reads or writes files, which is how automation remembers inventory, commands, or reports.
7. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import yaml
from concurrent.futures import ProcessPoolExecutor, wait
from datetime import datetime
from getpass import getpass
from netmiko import ConnectHandler


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


def ssh_conn(device_name, device_dict, cmd=None):
    with ConnectHandler(**device_dict) as net_connect:
        if cmd is None:
            return net_connect.find_prompt()
        else:
            output = net_connect.send_command(cmd)
            return (device_name, output)


# ... 47 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.27
# Source: netmiko_course/class8/collateral/cf_processes.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.28

Source: `netmiko_course/class8/collateral/cf_threads.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, yaml, concurrent.futures.ThreadPoolExecutor, concurrent.futures.wait, datetime.datetime, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `load_devices, ssh_conn`.
6. It reads or writes files, which is how automation remembers inventory, commands, or reports.
7. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import yaml
from concurrent.futures import ThreadPoolExecutor, wait
from datetime import datetime
from getpass import getpass
from netmiko import ConnectHandler


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


def ssh_conn(device_name, device_dict, cmd=None):
    with ConnectHandler(**device_dict) as net_connect:
        if cmd is None:
            return net_connect.find_prompt()
        else:
            output = net_connect.send_command(cmd)
            return (device_name, output)


# ... 45 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.28
# Source: netmiko_course/class8/collateral/cf_threads.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.29

Source: `netmiko_course/class8/collateral/cf_threads_asc.py`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, yaml, concurrent.futures.ThreadPoolExecutor, concurrent.futures.as_completed, datetime.datetime, getpass.getpass`.
2. Then it gets secrets from the environment or a password prompt, instead of hard-coding them.
3. Next, it builds a device connection dictionary and hands it to Netmiko's `ConnectHandler`.
4. After that, it sends a show command and saves the text that comes back from the device.
5. It defines reusable function(s): `load_devices, ssh_conn`.
6. It reads or writes files, which is how automation remembers inventory, commands, or reports.
7. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- A device dictionary is a contract: device type, host, username, and password must be correct.
- Show commands are read-only, so they are the safest first step in real network automation.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Replace the host with a lab-only device, load the password from an environment variable, and run a read-only command first.

**Source Preview**

```python
import os
import yaml
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from getpass import getpass
from netmiko import ConnectHandler


def load_devices(device_file="lab_devices.yml"):
    device_dict = {}
    with open(device_file) as f:
        device_dict = yaml.safe_load(f)
    return device_dict


def ssh_conn(device_name, device_dict, cmd=None):
    with ConnectHandler(**device_dict) as net_connect:
        if cmd is None:
            return net_connect.find_prompt()
        else:
            output = net_connect.send_command(cmd)
            return (device_name, output)


# ... 42 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.29
# Source: netmiko_course/class8/collateral/cf_threads_asc.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.30

Source: `netmiko_course/class8/collateral/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
  host: arista3.lasthop.io
# ... 46 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.30
# Source: netmiko_course/class8/collateral/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.31

Source: `netmiko_course/class9/collateral/test.txt`

**What This Example Is For**

It runs the same network task across multiple devices without waiting for one device at a time.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Concurrency saves time, but every result needs a clear success or failure record.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```text
# Script to detect memory leaks. It runs on the Polaris switch.
# $Id: memleak.tcl,v 1.63 2018/02/07 01:41:41 vpendyal Exp $
# Copyright (c) 2015-2019 by Cisco Systems, Inc.
# Author: Veeru Pendyala
#         2 Sep 2015
# Usage: 
#  Copy the tcl script to flash on switch
#  Execute 'tclsh' from IOS prompt
#  Prepare testbed for test (i.e. bringup your sessions etc.)
#  Execute 'memleak_baseline' to get a baseline of memory snapshot
#  Perform the tests like session leave/rejoin, roam etc.
#  Execute 'memleak_detect' to get summary of leak report
# Things to control:
#  What leak types to detect, by default all types enabled except ios_gd_leaks
#  and ios_gdchunk_leaks
#  Modify proc_list variables to add/remove processes from monitoring list
#  Change leak_threshold to whatever you need
#  Control standby flag whether need to monitor stdby procs or not

##########Global variables##########
#Base process list common to all platforms in polaris
set proc_list "ios fman_rp fman_fp repm dbm cli_agent smand plogd hman lman \
               nginx psd btman"
set leak_types "rss_leaks ios_gd_leaks ios_gdchunk_leaks ios_procmem_leaks \
# ... 3148 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.31
# Source: netmiko_course/class9/collateral/test.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.32

Source: `netmiko_course/class9/collateral/test2.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
no logging console
logging buffered 50000
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.32
# Source: netmiko_course/class9/collateral/test2.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.33

Source: `netmiko_course/class9/collateral/testx.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
Cisco IOS
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.33
# Source: netmiko_course/class9/collateral/testx.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.34

Source: `netmiko_course/.netmiko.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
# Dictionaries are devices
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
# ... 46 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.34
# Source: netmiko_course/.netmiko.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.35

Source: `netmiko_course/README.md`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```markdown
# netmiko_course
Netmiko Course / Python for Network Engineers
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.35
# Source: netmiko_course/README.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.36

Source: `netmiko_course/lab_devices.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
cisco3:
  device_type: cisco_xe
  host: cisco3.lasthop.io
  username: pyclass

cisco4:
  device_type: cisco_xe
  host: cisco4.lasthop.io
  username: pyclass

arista1:
  device_type: arista_eos
  host: arista1.lasthop.io
  username: pyclass

arista2:
  device_type: arista_eos
  host: arista2.lasthop.io
  username: pyclass

arista3:
  device_type: arista_eos
  host: arista3.lasthop.io
# ... 58 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.36
# Source: netmiko_course/lab_devices.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.37

Source: `netmiko_course/requirements.txt`

**What This Example Is For**

It shows how Python opens an SSH-style network session, sends commands, and then cleans up the connection.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Tests let you practice failure in a harmless place before a network change window.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
netmiko >= 3.3.0
pytest==6.0.2
pylama==7.7.1
black==20.8b1
ipdb>=0.13.3
ipython>=7.18.1
pyyaml
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.37
# Source: netmiko_course/requirements.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.38

Source: `netmiko_course/setup.cfg`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```ini
[metadata]
license_file = LICENSE

[pylama]
linters = mccabe,pep8,pyflakes
ignore = D203,C901
skip = .tox/*

[pylama:pep8]
max_line_length = 100
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.38
# Source: netmiko_course/setup.cfg
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.39

Source: `python_course_mar26/class4/concurrency/my_devices.py`

**What This Example Is For**

It repeats a network task over items such as devices, interfaces, commands, or retries.

**Explain It Like You Are New**

1. First, it brings in helper tools: `os, dotenv.load_dotenv`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
import os
from dotenv import load_dotenv


# This looks for a .env file and loads it
load_dotenv()

secret = os.environ["CHKP_EXPERT"]
admin_pass = os.environ["CHKP_ADMIN"]

chkpt_fw1 = {
    "host": "chkpnt-pod1.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
    "secret": secret,
}
chkpt_fw2 = {
    "host": "chkpnt-pod2.lasthop.io",
    "device_type": "checkpoint_gaia",
    "username": "admin",
    "use_keys": True,
    "key_file": "/home/kbyers/.ssh/eu-sshkey.pem",
# ... 32 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.39
# Source: python_course_mar26/class4/concurrency/my_devices.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.40

Source: `python_course_mar26/class1/exercises/func_ex/func_ex1.md`

**What This Example Is For**

It explains an exercise or workflow that the code examples are meant to practice.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Function Exercise1

Create a simple function named 'my_func' that takes no arguments. This function should print "Hello world".

Call this function three times.

Executing your Python program should produce the following output:

'''bash
$ python func_ex1.py 
Hello world
Hello world
Hello world
'''
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.40
# Source: python_course_mar26/class1/exercises/func_ex/func_ex1.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.41

Source: `python_course_mar26/class1/exercises/func_ex/func_ex1.py`

**What This Example Is For**

It wraps repeated network work in functions, so the same idea can be reused safely.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. It defines reusable function(s): `my_func`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#!/usr/bin/env python


def my_func():
    print("Hello world")


my_func()
my_func()
my_func()
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.41
# Source: python_course_mar26/class1/exercises/func_ex/func_ex1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.42

Source: `python_course_mar26/class1/exercises/func_ex/func_ex2.md`

**What This Example Is For**

It proves the script can run by printing a visible message, which is the first feedback loop before automating devices.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Function Exercise2

Create a function named "fw_func" that has one parameter "fw_name". The function body should just use an f-string to print out the value of the "fw_name" variable:

'''python
print(f"{fw_name=}")
'''

Call fw_func using a positional argument. Call fw_func using a named argument.

Your output should look similar to the following:

'''bash
$ python func_ex2.py 
fw_name='fw1'
fw_name='chkpnt-fw1'
'''
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.42
# Source: python_course_mar26/class1/exercises/func_ex/func_ex2.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.43

Source: `python_course_mar26/class1/exercises/func_ex/func_ex2.py`

**What This Example Is For**

It wraps repeated network work in functions, so the same idea can be reused safely.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. It defines reusable function(s): `fw_func`.
3. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#!/usr/bin/env python


def fw_func(fw_name):
    print(f"{fw_name=}")


fw_func("fw1")
fw_func(fw_name="chkpnt-fw1")
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.43
# Source: python_course_mar26/class1/exercises/func_ex/func_ex2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.44

Source: `python_course_mar26/class1/exercises/func_ex/func_ex3.py`

**What This Example Is For**

It wraps repeated network work in functions, so the same idea can be reused safely.

**Explain It Like You Are New**

1. The first line tells Unix-like systems which Python program should run this file.
2. First, it brings in helper tools: `rich.print`.
3. It defines reusable function(s): `fw_func`.
4. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Call the function with two different fake devices and compare the return values.

**Source Preview**

```python
#!/usr/bin/env python
from rich import print


def fw_func(name, ipaddr, os_version="R82"):
    print(f"{name=}")
    print(f"{ipaddr=}")
    print(f"{os_version=}")
    return f"{name}-{ipaddr}-{os_version}"


# Positional arguments
print("\nFunction call with positional arguments")
print("-" * 30)
ret_val = fw_func("chkpnt-pod99", "3.77.44.109", "R81.20")
print(f"\n{ret_val=}\n")

# Named arguments
print("\nFunction call with named arguments")
print("-" * 30)
ret_val = fw_func(
    os_version="R82",
    ipaddr="3.77.44.100",
    name="chkpnt-pod1",
# ... 21 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.44
# Source: python_course_mar26/class1/exercises/func_ex/func_ex3.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.45

Source: `python_course_mar26/class1/exercises/list_ex/locations.yml`

**What This Example Is For**

It stores inventory or settings in a human-readable file that scripts can load and reuse.

**Explain It Like You Are New**

1. This is data, not a program: it gives Python facts to work with.
2. Think of each key as a label on a box and each value as what is inside the box.
3. A script can load this file and use the values to decide which devices or commands to handle.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Structured data is easier for Python to trust than copied terminal text.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one fake device or object, then write a tiny loop that prints the fields you need.

**Source Preview**

```yaml
---
- Berlin
- Munich
- Cologne
- Frankfurt
- Hamburg
- Stuttgart
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.45
# Source: python_course_mar26/class1/exercises/list_ex/locations.yml
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.46

Source: `python_course_mar26/class3/exercises/fw_policy_ex1/chkpt_exceptions.py`

**What This Example Is For**

It bundles network data and behavior into an object, like making a small model of a device or session.

**Explain It Like You Are New**

1. It defines class blueprint(s): `ChkPntConfigError, ChkPntPolicyInstallError`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
class ChkPntConfigError(Exception):
    pass


class ChkPntPolicyInstallError(Exception):
    pass
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.46
# Source: python_course_mar26/class3/exercises/fw_policy_ex1/chkpt_exceptions.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.47

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/chkpt_exceptions.py`

**What This Example Is For**

It bundles network data and behavior into an object, like making a small model of a device or session.

**Explain It Like You Are New**

1. It defines class blueprint(s): `ChkPntConfigError, ChkPntPolicyInstallError`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
class ChkPntConfigError(Exception):
    pass


class ChkPntPolicyInstallError(Exception):
    pass
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.47
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/chkpt_exceptions.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.48

Source: `python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.md`

**What This Example Is For**

It makes a decision from network data, such as whether to act, skip, retry, or report.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
### Delete firewall rule exercise

Add an additional parameter to your 'cfg_fw_rule' function. This parameter should be named 'delete_rule' and should default to False.

If you call your cfg_fw_rule function and specify 'delete_rule=True', then the function will delete the specified firewall rule.

Recreate the code you used in the 'edit firewall rule' exercise except use your function to delete the specified firewall rule.

After the firewall rule has been deleted, publish your changes, and install your new firewall policy.
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.48
# Source: python_course_mar26/class3/exercises/fw_policy_ex2/fw_policy_delete_rule.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.49

Source: `python_course_mar26/README.md`

**What This Example Is For**

It explains an exercise or workflow that the code examples are meant to practice.

**Explain It Like You Are New**

1. This is an exercise or explanation file.
2. It tells the human what problem to solve before or after running the Python code.
3. In the book, this becomes the bridge between the idea and the hands-on network task.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```markdown
# python_course_mar26
Python Course March 2026
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.49
# Source: python_course_mar26/README.md
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.50

Source: `python_course_mar26/class1/libraries/lib_test1.py`

**What This Example Is For**

It proves the script can run by printing a visible message, which is the first feedback loop before automating devices.

**Explain It Like You Are New**

1. First, it brings in helper tools: `re, ipdb`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
import re
import ipdb  # noqa

ipdb.set_trace()
print(re.__file__)
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.50
# Source: python_course_mar26/class1/libraries/lib_test1.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.51

Source: `python_course_mar26/class1/libraries/lib_test2.py`

**What This Example Is For**

It is a small course example that supports the chapter's network automation idea.

**Explain It Like You Are New**

1. First, it brings in helper tools: `rich.print, re.search, ipdb`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
from rich import print
from re import search
import ipdb  # noqa

ipdb.set_trace()
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.51
# Source: python_course_mar26/class1/libraries/lib_test2.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.52

Source: `python_course_mar26/class1/libraries/sys_path_ex.py`

**What This Example Is For**

It proves the script can run by printing a visible message, which is the first feedback loop before automating devices.

**Explain It Like You Are New**

1. First, it brings in helper tools: `sys, rich.print`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Add one more device or interface to the list and predict how many lines print.

**Source Preview**

```python
"""Can't use ipdb for this since it alters sys.path."""
import sys
from rich import print

print("\nsys.path:")
print("-" * 30)
print(sys.path)
print()
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.52
# Source: python_course_mar26/class1/libraries/sys_path_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.53

Source: `python_course_mar26/class3/sets/set_ex.py`

**What This Example Is For**

It proves the script can run by printing a visible message, which is the first feedback loop before automating devices.

**Explain It Like You Are New**

1. First, it brings in helper tools: `rich.print`.
2. It prints something you can see, so you know the script actually ran.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
from rich import print

my_list = [1, 1, 3, 4, 7, 6, 7, 8, 3, 10]
my_list2 = [1, 1, 7, 6, 7, 8, 3, 10, 20, 45, 81, 99]

my_set1 = set(my_list)
print(type(my_set1))
print(my_set1)

my_set2 = set(my_list2)
print(my_set2)

union_sets = my_set1 | my_set2
print(union_sets)

intersect_sets = my_set1 & my_set2
print(intersect_sets)

set_diff1 = my_set1 - my_set2
print(f"{my_set1=}")
print(f"{my_set2=}")
print(set_diff1)

set_diff2 = my_set2 - my_set1
# ... 3 more line(s) in source file
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.53
# Source: python_course_mar26/class3/sets/set_ex.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.54

Source: `python_course_mar26/lib_class4/chkpt_exceptions.py`

**What This Example Is For**

It bundles network data and behavior into an object, like making a small model of a device or session.

**Explain It Like You Are New**

1. It defines class blueprint(s): `ChkPntConfigError, ChkPntPolicyInstallError`.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.
- Network scripts must expect failure: DNS, TCP, authentication, prompts, and timeouts can all break.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```python
class ChkPntConfigError(Exception):
    pass


class ChkPntPolicyInstallError(Exception):
    pass
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.54
# Source: python_course_mar26/lib_class4/chkpt_exceptions.py
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

#### Twin-bridges source map 20.55

Source: `python_course_mar26/work/notes.txt`

**What This Example Is For**

It provides command text, sample output, hosts, or notes that another script can consume.

**Explain It Like You Are New**

1. This supporting file provides text, templates, commands, or sample output for the scripts.

**Foundational Ideas**

- The Python idea here is `package scripts, run checks, log sessions, and prepare production workflows`.
- Networking code is just normal Python with more serious inputs and outputs.

**How It Connects to This Chapter**

`local script -> config/logging/tests -> scheduled run -> operational report`

**Try This Next**

Change one value, predict the output, then run the cell or script and compare.

**Source Preview**

```text
show-arp
```

In [ ]:
# Practice rewrite for twin-bridges source map 20.55
# Source: python_course_mar26/work/notes.txt
# Rewrite one idea from the source with simulated data before using a real device.
simulated_device = {"hostname": "edge-sw1", "platform": "cisco_ios", "reachable": True}
print(f"Plan for {simulated_device['hostname']}: inspect the source pattern, then dry-run it.")

### Chapter 20: Operating and Deploying Automation Practice Checkpoint

1. Pick one safe example from this chapter and rename the devices to match your lab.
2. Write the command or API action you would run in a dry-run variable before connecting to anything.
3. Add one validation check that would stop the script from making a bad change.
4. Compare the authored bridge example with the assigned twin-bridges source files and note what becomes real-device-specific.

## Final Study Routine

1. Name the Python concept from PCC.
2. Identify the matching network object: device, interface, command, API response, config, or test.
3. Run the safe simulated cell.
4. Read the assigned twin-bridges source map.
5. Rewrite the example against fake data.
6. Only then adapt it to a lab device with environment-based credentials and logging.